<a href="https://colab.research.google.com/github/leongriner-portfolio/synth-haem/blob/main/synth_haem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

# Environment Setup (Run on Session Start)

As the simulation and analyses scripts need to run in the Google Colab environment, a number of resource constraints and workflows needs to be addressed.

The main issue that needs to be addressed is sessions are wiped periodically, so the environment needs to built/loaded each time. Due to space constraints, and large intermediate file sizes, we need to write to Google Drive periodically. While it would be possible to just write to local storage, it's faster and safer to just retrieve environment, reference, simulation and data files from the PROJECT (Google Drive) storage folder.

However, it's important to check that environment settings are all account for, directory structure is consistent etc.

The Environment Setup block of cells will do the following:
*   Specify settings, creating consistent directory structure and loads data, and load this into environment
*   Mount Google Drive storage and checks mounting
*   Copy stored data, reference files, etc into local
*   Sets tools in environment.yml, installs and builds environment
*   Checks versions, sets lock file and caches that environment so that it's consistent across runs/re-loads after session wipes
*   Saves local (WORK) files into Drive Storage (PROJECT)  

While these steps to maintain consistency, many are time-consuming and there are opportunites for error:
*   Accidently running the Environment Setup block of cells would overwrite local files that had been generated, if older version are in storage
*   Building the environment is time consuming and there are opportunities for version changes in between builds, which might result in unexpected behaviour/create issues with reproducibility

Consequently, re-running the block, after it's been run once effectively does nothing:
*   Mounting does nothing if already mounted
*   Copying Drive files does nothing unless a FORCE_RECOPY switch is set to TRUE (as restoring stored files if wanted is desirable behaviour).
*   Writing environment.yml does nothing if it already exists.
*   Instead of rebuilding from environment.yml, loads the cache, failing that uses lockfile. However, once cache is already generated, uses that (unless make_cache switch set to True)



## Settings, Load Data from Drive, Check Machine config

In [23]:
import os

# Create SETTINGS dictionary (dict)
# Reminder how dicts work here: {} used to set values, and subsequent [] used to ref
# First pass sets keys, then subsequent SETTINGS[key] = f{}

SETTINGS = {
    #Google drive folder
    "PROJECT": "/content/drive/MyDrive/Colab Notebooks/Project Storage/Bioinformatics Portfolio",
    #local dir
    "WORK": "/content/my_project",
    #locally-installed tools
    "TOOLS": "/content/tools",
    # Copy markers: recording when data last moved between Drive (PROJECT) and Local (WORK).
    # Deliberately OUTSIDE WORK, so saving WORK to Drive never carries them up.
    # Lost at wipe so new session will start with fresh copy markers.
    "MARKERS_DIR": "/content/markers",
    #mamba envir - note do not rename, name not ad hoc used by micromamba
    #unless remapping with os.environ["MAMBA_ROOT_PREFIX"] = SETTINGS[WHATEVER]
    "MAMBA_ROOT_PREFIX": "/content/micromamba",
    #project environment name
    "ENV_NAME": "bioinf_portfolio",
    #BAMsurgeon commit this was checked against - don't want it to break from upstream changes
    "BAMSURGEON_COMMIT": "d9448a5"
    }

# Derived paths, built from the settings above.
SETTINGS["ENV_DIR"] = f"{SETTINGS['PROJECT']}/env"                       # env files on Drive
SETTINGS["ENV_YML"] = f"{SETTINGS['ENV_DIR']}/environment.yml"           # what to install
SETTINGS["ENV_LOCK"] = f"{SETTINGS['ENV_DIR']}/bioinf_portfolio.lock.yml"      # exact versions written after the first build
SETTINGS["ENV_CACHE"] = f"{SETTINGS['ENV_DIR']}/bioinf_portfolio-env.tar.gz"   # optional packed copy of the built environment
SETTINGS["MICROMAMBA"] = f"{SETTINGS['TOOLS']}/bin/micromamba"                      # the micromamba program
SETTINGS["ENV_BIN"] = f"{SETTINGS['MAMBA_ROOT_PREFIX']}/envs/{SETTINGS['ENV_NAME']}/bin"  # the environment's tools
SETTINGS["LOGS_DIR"] = f"{SETTINGS['WORK']}/output/logs"                 # logs, local; later section saves them to Drive
SETTINGS["BAMSURGEON_DIR"] = f"{SETTINGS['TOOLS']}/bamsurgeon"                      # BAMSurgeon's code (re-cloned each session)
#note that version of BAMsurgeon that uses the spike-in is not packaged yet so code needs to be run
SETTINGS["DRIVE_COPY_MARKER"] = f"{SETTINGS['MARKERS_DIR']}/drive_to_work.txt"      # when Drive was last copied to WORK (section 2)
SETTINGS["WORK_COPY_MARKER"] = f"{SETTINGS['MARKERS_DIR']}/work_to_drive.txt"       # when WORK was last saved to Drive (save cells, later)


#export SETTINGS as environment variables
os.environ.update(SETTINGS)

# Keys later sections rely on, written out separately from SETTINGS so that a
# line missing from SETTINGS is caught here, not as an error sections later.
# Add a key here whenever a new section starts using one.
REQUIRED = [
    # Drive <-> WORK copying and saving sections
    "PROJECT", "WORK", "MARKERS_DIR", "DRIVE_COPY_MARKER", "WORK_COPY_MARKER",
    # micromamba and the environment sections
    "TOOLS", "MICROMAMBA", "MAMBA_ROOT_PREFIX", "ENV_NAME", "ENV_DIR",
    "ENV_YML", "ENV_LOCK", "ENV_CACHE", "ENV_BIN",
    # BAMSurgeon section
    "BAMSURGEON_DIR", "BAMSURGEON_COMMIT",
    # Logs section
    "LOGS_DIR",
]

# Checks if items in REQUIRED is missing from the environment.
missing = [key for key in REQUIRED if key not in os.environ]
if missing:
    raise RuntimeError(f"Missing from the environment: {missing}. Check section 1's SETTINGS.")
print(f"All {len(REQUIRED)} required settings exported.")


All 17 required settings exported.


NOTE: ONLY RUN AT START

Project will write to local disk, as it is faster, but will store to Drive to keep outputs between wipes.

In [2]:
import os
from pathlib import Path
import shutil
import time
from datetime import datetime, timezone
from google.colab import drive

# Mount Drive. If it is already mounted this session, this does nothing
# because force_remount defaults to False. To force remount call
# drive.mount("/content/drive", force_remount=True).
drive.mount("/content/drive")

#Check if mounted to prevent create issue writing fake dir structure
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive is not mounted. Rerun this cell.")

Path(SETTINGS["PROJECT"]).is_dir() #checks Project exists in drive.

# Paths come from section 1: Setup - Environment, so they can't change
# Not explained in Section 1 but writing to local for speed reasons but using
# Google Drive as Project path for long term storage between Colab wipes
# This creates an issue where running this cell accidently would trigger re-copy
# This would over-write the existing WORK directory with old data.
# Copy-handling via FORCE_RECOPY and Copy Markers (DRIVE- and WORK-COPY-MARKER)
# Only change FORCE_RECOPY = True if over-writing data

FORCE_RECOPY = False

PROJECT_path = Path(SETTINGS["PROJECT"])
WORK_path = Path(SETTINGS["WORK"])
DRIVE_COPY_MARKER_path = Path(SETTINGS["DRIVE_COPY_MARKER"])  # written below, after each PROJECT -> WORK copy
WORK_COPY_MARKER_path = Path(SETTINGS["WORK_COPY_MARKER"])    # written by the save cells, after each WORK -> Drive copy

Path(SETTINGS["MARKERS_DIR"]).mkdir(parents=True, exist_ok=True)

# COPY HANDLING:
# Define the creating markers, used when stamping/reading copies from PROJECT and WORK
def stamp_marker(marker):
  """Write current UTC time into marker file for read_marker to read it back
  The save cells in later sections call this too, for WORK_COPY_MARKER_path.
  """
  marker.write_text(datetime.now(timezone.utc).isoformat(timespec="seconds"))

def read_marker(marker):
    """Return Time UTC, N min ago from a marker, or None if there's no marker yet."""
    if not marker.exists():
        return None
    when = datetime.fromisoformat(marker.read_text().strip())
    minutes = (datetime.now(timezone.utc) - when).total_seconds() / 60
    return f"{when:%Y-%m-%d %H:%M} UTC, {minutes:.0f} min ago"

#Check if a fresh copy or if FORCE_RECOPY = TRUE, otherwise don't copy PROJECT files.
if DRIVE_COPY_MARKER_path.exists() and not FORCE_RECOPY:
    # Not a fresh session: WORK may hold changes not yet saved to Drive, so don't copy.
    print("Did not copy from Drive: this session has already loaded the project.")
    print(f"  Last loaded from Drive:  {read_marker(DRIVE_COPY_MARKER_path)}")
    # 'or' gives 'not yet saved' when read_marker returns None (no save yet).
    # Reminder to self on how this works: 'None' returned from read_marker counts as
    # False (and string UTC date = True) when evaluating the f"'A' or 'B'.""
    print(f"  Last saved to Drive:     {read_marker(WORK_COPY_MARKER_path) or 'not saved yet'}")
    print("To copy anyway, set FORCE_RECOPY = True and re-run this cell.")
    print("A forced copy replaces local files with their Drive versions.")
else:
    # Fresh session (no Drive marker yet), or FORCE_RECOPY = True.
    start = time.time()
    # dirs_exist_ok=True lets copytree copy into WORK when it already exists.
    # Files in both places get the Drive version; files only in WORK are left alone.
    # ignore skips env/ (any file or folder named exactly env): its files are used
    # straight from Drive (section 1), and keeping them out of WORK means section 11
    # can never overwrite them with stale copies. It also skips *.partial files:
    # leftovers from an interrupted save (section 11).
    shutil.copytree(PROJECT_path, WORK_path, dirs_exist_ok=True, ignore=shutil.ignore_patterns("env", "*.partial"))
    # Stamp the marker only after the copy succeeds, so a failed copy runs again next time.
    stamp_marker(DRIVE_COPY_MARKER_path)
    print(f"Copied {PROJECT_path} -> {WORK_path} in {time.time() - start:.0f} s")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied /content/drive/MyDrive/Colab Notebooks/Project Storage/Bioinformatics Portfolio -> /content/my_project in 115 s


In [3]:
%%bash
# Check Machine:
# nproc: number of CPUs available (sets how many threads to use later. Expecting 2).
echo "CPUs: $(nproc)"
# free -h: memory human-readable. bwa index needs about 5 GB.
free -h
# df -h: free space on the local disk (where the working files go).
df -h /content
# and Drive (Reminder: || in bash works to run fallback if non-zero exit code)
df -h /content/drive 2>/dev/null || echo "Drive not mounted yet: run section 2 first."

CPUs: 2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.5Gi       246Mi       3.0Mi        11Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   19G   89G  18% /
Filesystem      Size  Used Avail Use% Mounted on
drive           100G   89G   12G  89% /content/drive


## Dirs and Env, Micromamba install, load environment

Creates the folders this workflow needs, including env/ on Drive, then writes environment.yml there (only if it isn't there yet).

It lists every package, plus r-base for the results table and pip/setuptools for installing BAMSurgeon.

This lives on Drive.

**Reminder to self:**
In Colab, each %%bash cell starts a fresh shell.
Without pipefail, default behaviour is to continue past errors, treat unset variables as empty text, and checks pipeline if last command worked.

Conversely:

```
set -e            # stop at the first failing command
set -u            # treat an unset variable as an error
set -o pipefail   # a pipeline fails if any command in it fails
```

**BAMsurgeon:** tools for adding mutations to existing .bam files, used for testing mutation callers. Importantly, an August 2026 commit allows for spike for low VAF mutations.

Bioconda’s only build is noarch/bamsurgeon-1.4.1-pyhdfd78af_0.tar.bz2, uploaded Jun 13, 2024. It packages the GitHub release from August 24, 2022, described as a bugfix release for 1.4, read pairs were not maintained in spike-in.

Section 1 pins GitHub commit d9448a5, from August 2026. That code has never been released as a new version, and its pyproject.toml still says version = "1.4". So the newer code carries the smaller number. This commit adds the features the spike-in step relies

In [ ]:
%%bash
set -euo pipefail

#-p doesn't complain if parent exists
# $TOOLS/bin and MAMBA_ROOT_PREFIX are local
# ENV_DIR is on Drive

mkdir -p "$TOOLS/bin" "$MAMBA_ROOT_PREFIX" "$ENV_DIR"

#write environment.yml only if not on drive (i.e., should only do something on first run)
if [ ! -f "$ENV_YML" ]; then #if not a file...
 cat > "$ENV_YML" <<'EOF'
# Software for the synthetic haem panel dataset.
# Channel order matters: conda-forge first, then bioconda, as bioconda expects.
name: bioinf_portfolio
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11      # pinned so BAMSurgeon's pip install has a known base
  - pip              # for installing BAMSurgeon from its GitHub source
  - setuptools=84.2  # tested with BAMSurgeon d9448a5; newer versions may reject its licence format after Feb 2027
  - samtools         # index FASTA; sort, index, subsample, merge BAMs; BAM to FASTQ
  - htslib           # bgzip and tabix
  - bcftools         # sort, normalise, filter, compare, query VCFs
  - bedtools         # pad, merge, intersect, extract regions
  - bwa              # index the reference; align reads
  - art              # ART read simulator (art_illumina)
  - wgsim            # BAMSurgeon dependency (its SV engine)
  - pysam            # BAMSurgeon dependency
  - mosdepth         # depth over targets
  - picard           # interval lists; capture metrics
  - gatk4            # dictionary, MarkDuplicates, Mutect2, FilterMutectCalls, HaplotypeCaller
  - rtg-tools        # reference SDF; vcfeval scoring
  - jq               # pull fields out of JSON replies (Variant Recoder)
  - curl             # downloads and web API queries
  - r-base           # results table (base R only)
EOF
  echo "Wrote $ENV_YML (on Drive, so later sessions keep it)."
else
  echo "Using environment.yml from Drive (edit it there; delete it to regenerate)."
fi

# Show what will be installed.
cat "$ENV_YML"

Using environment.yml from Drive (edit it there; delete it to regenerate).
# Software for the synthetic haem panel dataset.
# Channel order matters: conda-forge first, then bioconda, as bioconda expects.
name: bioinf_portfolio
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11      # pinned so BAMSurgeon's pip install has a known base
  - pip              # for installing BAMSurgeon from its GitHub source
  - setuptools       # BAMSurgeon's manual asks for setuptools >= 61
  - samtools         # index FASTA; sort, index, subsample, merge BAMs; BAM to FASTQ
  - htslib           # bgzip and tabix
  - bcftools         # sort, normalise, filter, compare, query VCFs
  - bedtools         # pad, merge, intersect, extract regions
  - bwa              # index the reference; align reads
  - art              # ART read simulator (art_illumina)
  - wgsim            # BAMSurgeon dependency (its SV engine)
  - pysam            # BAMSurgeon dependency
  - mosdepth         # depth ov

In [ ]:
%%bash
set -euo pipefail

# skip if already has it this session.
if [ ! -x "$MICROMAMBA" ]; then
  cd "$TOOLS"
  # curl options: -L follow redirects, -s silent. The download is a .tar.bz2
  # archive; tar -x extracts, -v lists files, -j means bzip2, and
  # bin/micromamba extracts only that one file (to $TOOLS/bin/micromamba).
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

# Print the version to confirm it runs.
"$MICROMAMBA" --version

bin/micromamba
2.10.0


## Build/restore environment

First pass: build from .yml. Later passes build from either lock file or unpack cached .tar.gz environment stored in Drive.

If version issues are occurring, change the versions in the environment.yml and rebuild from there (deleting the lock and cache from Drive first).

To specify version, change the generic package name:

In environment.yml | Matches
-------------------|------------------
samtools=1.20 | any 1.20.x release (1.20, 1.20.1, …) |
samtools==1.20 |	exactly 1.20
samtools>=1.20,<1.22 | a range
samtools=1.20=h50ea8bc_0 | one exact build (rarely needed; this one made up)

To see which versions exist before pinning:
`"$MICROMAMBA" search -c conda-forge -c bioconda samtools`




In [4]:
%%bash
set -euo pipefail

ENV_PATH="$MAMBA_ROOT_PREFIX/envs/$ENV_NAME"
# SECONDS is a bash built-in counter, used here to time the step.
SECONDS=0

if [ -d "$ENV_PATH" ]; then
  echo "Environment already present in this session: $ENV_PATH"

elif [ -f "$ENV_CACHE" ]; then
  echo "Restoring the cached environment..."
  # Unpack into MAMBA_ROOT_PREFIX's parent folder (/content), where we'll later pack
  # it from. Conda environments contain hard-coded paths, so they only work at
  # the path they were built in. -x extract, -z gzip, -f the archive file.
  tar -C "$(dirname "$MAMBA_ROOT_PREFIX")" -xzf "$ENV_CACHE"

elif [ -f "$ENV_LOCK" ]; then
  echo "Rebuilding from the lock file (exact versions from the first build)..."
  # create: make a new environment. -n: its name. -f: the file listing
  # packages. -y: answer yes to the confirmation prompt. micromamba puts it
  # under MAMBA_ROOT_PREFIX, set in Cell 1: Settings.
  "$MICROMAMBA" create -y -n "$ENV_NAME" -f "$ENV_LOCK"

else
  echo "First build from environment.yml (latest versions)..."
  "$MICROMAMBA" create -y -n "$ENV_NAME" -f "$ENV_YML"
fi

# Check the environment landed where ENV_BIN) expects it.
# -x: the file exists and can be run.
if [ ! -x "$ENV_BIN/python3" ]; then
  echo "ERROR: no environment at $ENV_PATH."
  echo "Check that section 1 ran in this session and sets MAMBA_ROOT_PREFIX."
  exit 1
fi

echo "Done in $((SECONDS / 60)) min $((SECONDS % 60)) s."

Restoring the cached environment...
Done in 0 min 51 s.


**Make PATH variable:**

Made after tools are in environment's bin, following environment being built/loaded.

In [20]:
import os
import shutil

env_bin = os.environ["ENV_BIN"]

# Prepend the env's bin folder to PATH (only once, even if this cell is re-run).
if not os.environ["PATH"].startswith(env_bin):
    os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

# Check that the tool resolves to environment's bin folder. Uses
for tool in ["python3", "samtools", "gatk"]:
   found = shutil.which(tool)
   if found and found.startswith(env_bin + "/"): # "/" at end to avoid statswith being fooled
      print(f"Confirmed: {tool} -> {found}")
   else:
      raise RuntimeError(f"{tool} resolves to {found}, not the environment.")


Confirmed: python3 -> /content/micromamba/envs/bioinf_portfolio/bin/python3
Confirmed: samtools -> /content/micromamba/envs/bioinf_portfolio/bin/samtools
Confirmed: gatk -> /content/micromamba/envs/bioinf_portfolio/bin/gatk


Installing BAMsurgeon (if required) - takes ~30s

In [6]:
%%bash
set -euo pipefail

# Clone only if this session doesn't have it yet. Remove --quiet if needing progress
if [ ! -d "$BAMSURGEON_DIR/.git" ]; then
  git clone --quiet https://github.com/adamewing/bamsurgeon.git "$BAMSURGEON_DIR"
fi
cd "$BAMSURGEON_DIR"

# Switch to the exact commit the doc was checked against.
git checkout --quiet "$BAMSURGEON_COMMIT"
echo "BAMSurgeon at commit: $(git rev-parse --short HEAD)"

# ---- Install BAMSurgeon into the environment -----------------------------
#
# Why there's a build step at all:
#   BAMSurgeon arrives as source code, not a ready-made package, so pip has to
#   build it before installing it. pip doesn't do the build itself: it hands
#   the job to the build tool the project names in its pyproject.toml:
#       [build-system]
#       requires = ["setuptools>=61"]
#   i.e. "build me with setuptools, version 61 or newer".
#
# What pip does by default ("build isolation"):
#   It makes a throwaway environment (the /tmp/pip-build-env-... path in the
#   old warnings), downloads the NEWEST setuptools from PyPI into it, builds
#   there, then deletes it. Our environment's own setuptools is ignored.
#   Two problems with that:
#   1. The build tool can change between sessions without us knowing, and the
#      version used is never recorded (it isn't in the lock file).
#   2. Recent setuptools warns that BAMSurgeon's pyproject.toml declares its
#      licence in an outdated format (license = { file = "LICENSE.txt" }
#      instead of a short identifier such as license = "MIT"), and says
#      support for it ends after 2027-02-18. A setuptools released after that
#      date could refuse to build BAMSurgeon, and this cell would start
#      failing with nothing changed on our side.
#
# The fix, --no-build-isolation:
#   pip skips the throwaway environment and builds with the setuptools already
#   installed in OUR environment: the one micromamba installed from
#   environment.yml and the lock file records (84.2.1 in the current build).
#   Same build tool every session, nothing extra downloaded, and no future
#   setuptools release can affect it.
#   The catch: pip no longer fetches build tools for us, so setuptools must
#   already be in the environment. It is, because environment.yml lists it.
#   (If it ever went missing, this line would fail with an error about
#   setuptools or the "build backend".)
#   You may still see the licence warning: setuptools 84 prints it too. It is
#   only a warning, and with this setuptools version fixed it can't become an
#   error.
#
# Why the old 'pip install --upgrade pip setuptools' line is gone:
#   The environment already has pip and setuptools far newer than BAMSurgeon
#   needs. Upgrading them with pip would replace the copies micromamba
#   installed, so the environment would no longer match the lock file.
#
# The rest of the command:
#   python3 -m pip   run the pip belonging to the environment's Python (first
#                    on PATH since section 7), not Colab's own pip
#   --quiet          hide progress output; errors and warnings still show
#   -e               "editable": installs the bamsurgeon command, but it runs
#                    the code in this folder rather than a copy, as the
#                    manual's install instructions do
#   .                what to install: the current folder ($BAMSURGEON_DIR, from
#                    the cd above), which holds pyproject.toml
python3 -m pip install --quiet --no-build-isolation -e .

# BAMSurgeon's own check for samtools, bwa, wgsim and pysam.
# -O is how its README runs the script (it only disables Python assert checks).
python3 -O scripts/check_dependencies.py

BAMSurgeon at commit: d9448a5
All dependencies are satisfied


  /content/micromamba/envs/bioinf_portfolio/lib/python3.11/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: BuildBackendWarning: `project.license` as a TOML table is deprecated
  !!
  
          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
  
          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
          ********************************************************************************
  
  !!
    corresp(dist, value, root_dir)
  /content/micromamba/envs/bioinf_portfolio/lib/python3.11/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: BuildBackendWarning: `project.

## Check versions, Write lockfile, Cache environment

In [7]:
%%bash
# No 'set -e' here: several tools exit with an error code when asked for help
# or a version, and that shouldn't stop the cell.
set -uo pipefail

# v LABEL COMMAND: print a heading, then the first 3 lines of the command's
# output (stderr included, since some tools print versions there).
v() {
  local label="$1"; shift
  echo "== $label"
  { bash -c "$*" 2>&1 || true; } | head -n 3
}

# The version list goes in WORK's output/logs (LOGS_DIR, section 1), like every
# other log. Section 11 saves it to Drive.
mkdir -p "$LOGS_DIR"
VERSIONS="$LOGS_DIR/tool_versions.txt"
{
  echo "Recorded: $(date -u '+%Y-%m-%d %H:%M UTC')"
  v "python"     "python3 --version"
  v "micromamba" "\"$MICROMAMBA\" --version"
  v "samtools"   "samtools --version"
  v "htslib"     "bgzip --version"
  v "bcftools"   "bcftools --version"
  v "bedtools"   "bedtools --version"
  v "bwa"        "bwa 2>&1 | grep -i '^version'"            # bwa prints its version in the usage text
  v "art"        "art_illumina 2>&1 | grep -i -m1 'version'" # same for ART
  v "wgsim"      "wgsim 2>&1 | grep -i -m1 'version'"
  v "mosdepth"   "mosdepth --version"
  v "picard"     "picard SortSam --version"
  v "gatk"       "gatk --version 2>&1 | grep -m1 'Genome Analysis Toolkit'"   # skip the launcher's preamble
  v "rtg-tools"  "rtg version"
  v "java"       "java -version"
  v "jq"         "jq --version"
  v "curl"       "curl --version"
  v "R"          "R --version"
  v "bamsurgeon commit" "git -C \"$BAMSURGEON_DIR\" rev-parse --short HEAD"   # the exact code installed
} | tee "$VERSIONS"   # tee prints to the screen AND writes the file
echo "Versions written to $VERSIONS (section 11 saves them to Drive)."

# Fail loudly if any command is missing from PATH.
missing=()
for tool in python3 samtools bgzip tabix bcftools bedtools bwa art_illumina wgsim \
            mosdepth picard gatk rtg java jq curl Rscript bamsurgeon; do
  command -v "$tool" > /dev/null || missing+=("$tool")
done
if [ ${#missing[@]} -gt 0 ]; then
  echo "MISSING: ${missing[*]}"
  exit 1
fi
echo "All tools found."

# Write the lock file once, after the first successful build. ENV_LOCK is in
# env/ on Drive (section 1), so it's saved there directly.
if [ ! -f "$ENV_LOCK" ]; then
  # env export lists every installed package with its exact version and build.
  "$MICROMAMBA" env export -n "$ENV_NAME" > "$ENV_LOCK"
  echo "Wrote the lock file to Drive: $ENV_LOCK"
fi

Recorded: 2026-11-22 22:41 UTC
== python
Python 3.11.18
== micromamba
bash: line 1: /content/tools/bin/micromamba: No such file or directory
== samtools
samtools 1.24
Using htslib 1.24
Copyright (C) 2026 Genome Research Ltd.
== htslib
bgzip (htslib) 1.24
Copyright (C) 2026 Genome Research Ltd.
== bcftools
bcftools 1.24
Using htslib 1.24
Copyright (C) 2026 Genome Research Ltd.
== bedtools
bedtools v2.31.1
== bwa
Version: 0.7.19-r1273
== art
          Q Version 2.5.8 (June 6, 2016)       
== wgsim
Version: 1.24
== mosdepth
mosdepth 0.3.14
== picard
Version:3.5.0
== gatk
The Genome Analysis Toolkit (GATK) v4.6.2.0
== rtg-tools
Product: RTG Tools 3.13
Core Version: c27844a5bb (2025-05-27)
RAM: 11.4GB of 12.6GB RAM can be used by rtg (90%)
== java
openjdk version "17.0.18-internal" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18-internal+0-adhoc.rattler.src)
OpenJDK 64-Bit Server VM (build 17.0.18-internal+0-adhoc.rattler.src, mixed mode, sharing)
== jq
jq-1.8.2
== curl
curl 8.23.0 (x

### OPTIONAL: Cache Environment

Takes ~3 mintes

In [ ]:
%%bash
set -euo pipefail

# Failsafe: only pack the cache when this is true. Set it back to false
# afterwards, so "Run all" never re-packs it by accident. (Bash has no
# True/False: the convention is the plain words true and false.)
MAKE_CACHE=false

if [ "$MAKE_CACHE" != true ]; then
  echo "Cache not written: MAKE_CACHE is false. Set it to true and re-run to pack the environment."
  exit 0    # end the cell here, successfully
fi

# Split MAMBA_ROOT_PREFIX (/content/micromamba) into its parent (/content) and its
# own name (micromamba), so the archive holds paths section 6 can unpack to
# the same place.
PARENT="$(dirname "$MAMBA_ROOT_PREFIX")"
NAME="$(basename "$MAMBA_ROOT_PREFIX")"

# ENV_CACHE is in env/ on Drive (section 1), so this writes straight there.
# It writes under a temporary name (.partial) and renames only once tar has
# finished. If the write is interrupted, only the .partial file is left:
# section 6 looks for the exact cache name, so it ignores it and falls back to
# the old cache or the lock file. Writing straight to the real name would wipe
# the old cache at the start and could leave a half-written one for section 6.
#   -C "$PARENT"          pack paths relative to the parent folder
#   --exclude="$NAME/pkgs" skip micromamba's download cache, not needed to run
#   -c create   -z gzip   -f output file
tar -C "$PARENT" --exclude="$NAME/pkgs" -czf "$ENV_CACHE.partial" "$NAME"
# set -e (top of the cell) means this only runs if tar succeeded.
mv "$ENV_CACHE.partial" "$ENV_CACHE"

# -h: human-readable size.
ls -lh "$ENV_CACHE"

Cache not written: MAKE_CACHE is false. Set it to true and re-run to pack the environment.


# Save WORK (Local) to PROJECT (Drive) storage

Run this whenever there's work worth saving (before the session ends): anything only in WORK is lost then. It isn't part of the start-of-session run.

Notes on behaviour:

*   Off by default. Set SAVE_TO_DRIVE = True to save, and back to False afterwards. When it's off, the cell only reports when you last saved.
*   Copies only what changed. Files whose Drive copy has the same size and modification time are skipped, so large BAMs aren't re-copied on every save. This relies on Drive keeping modification times; if it doesn't, every file is copied each time, which is slower but still correct. A second save straight after the first should report almost everything as unchanged.
*   Doesn't leave a half-written file on Drive. Each file is copied under a temporary .partial name and renamed only once it's complete, so if a copy fails, Drive keeps its previous version.
*   Never deletes anything on Drive, and skips env/ and .partial files.
*   Reports failures and stops. If anything can't be saved, it lists what (up to 20 items), leaves the save marker unchanged and stops with an error. Fix the cause and re-run: only what's still missing or changed gets copied.
*   Stops if Drive isn't mounted, so a save can't land in a fake Drive folder on the VM's own disk.

**If a save fails:**

Error says | Cause | Fix
-----------|----------|------
No space left on device or quota |  Drive is full | Free space (empty Drive's Bin too: deleted files still count), then re-run
Transport endpoint is not connected, or nearly every file fails | The Drive mount dropped | drive.mount("/content/drive", force_remount=True), then re-run
Input/output error, timeouts | A temporary Drive hiccup | Usually just re-run

In [58]:
from posix import truncate
from pathlib import Path
import os
import shutil
import time

# Save WORK back to Drive (PROJECT). Run whenever there's work worth keeping,
# and always before the session ends: anything only in WORK is lost then.
# stamp_marker and read_marker come from section 2. After Runtime -> Restart
# session, re-run sections 1 and 2 first (section 2 won't copy again).

# Failsafe: only save when this is True. Set it back to False afterwards,
# so "Run all" never saves by accident.
SAVE_TO_DRIVE = True

PROJECT_path = Path(SETTINGS["PROJECT"])
WORK_path = Path(SETTINGS["WORK"])
WORK_COPY_MARKER_path = Path(SETTINGS["WORK_COPY_MARKER"])

# Tally of what the save did, filled in by copy_changed_safely below.
counts = {"copied": 0, "unchanged": 0}

def copy_changed_safely(src, dst):
    """Copy one file from WORK to Drive. copytree calls this for every file.

    1. Skip it if Drive already has an identical copy (same size and
       modification time), so unchanged files, large BAMs included, aren't
       re-copied on every save.
    2. Otherwise copy it to dst.partial and rename that over dst once it's
       complete, so a failed copy never leaves a half-written file under the
       real name: Drive keeps its previous version.
    """
    if os.path.exists(dst):
        s, d = os.stat(src), os.stat(dst)
        # copy2 keeps a file's modification time, so a file unchanged since it
        # was copied matches its Drive copy. The 2-second leeway allows for
        # Drive storing times less precisely. If Drive doesn't keep the times
        # at all, nothing matches and every file is copied: slower, still correct.
        if s.st_size == d.st_size and abs(s.st_mtime - d.st_mtime) < 2:
            counts["unchanged"] += 1
            return dst
    tmp = f"{dst}.partial"
    shutil.copy2(src, tmp)   # the slow part; if it fails, dst is untouched
    os.replace(tmp, dst)     # a rename: the real name never points at a half-written file
    counts["copied"] += 1
    return dst


if not SAVE_TO_DRIVE:
    print("Not saved: SAVE_TO_DRIVE is False. Set it to True and re-run this cell to save.")
    print(f"  Last saved to Drive:     {read_marker(WORK_COPY_MARKER_path) or 'not saved yet'}")
else:
    # Stop if Drive isn't mounted, so the copy can't land in a fake Drive folder
    # on the VM's own disk.
    if not os.path.ismount("/content/drive"):
        raise RuntimeError("Google Drive is not mounted. Run section 2 first.")

    start = time.time()
    try:
        # dirs_exist_ok=True copies into the existing Drive folder. Nothing on
        # Drive is deleted: a file you delete in WORK stays on Drive until you
        # delete it there too.
        # ignore skips env/ (any file or folder named exactly env): its files live
        # on Drive and are written there directly, so Drive's copy is always the
        # current one. It also skips *.partial files, so leftovers from an
        # interrupted save are never copied around.
        shutil.copytree(WORK_path, PROJECT_path, dirs_exist_ok=True,
                        copy_function=copy_changed_safely,
                        ignore=shutil.ignore_patterns("env", "*.partial"))
    except shutil.Error as e:
        # copytree keeps going past failures and reports them all at the end:
        # e.args[0] is a list of (source, destination, reason).
        failed = e.args[0]
        print(f"SAVE INCOMPLETE: {len(failed)} item(s) were not saved to Drive.")
        for src, dst, why in failed[:20]:   # first 20, so a dropped mount can't flood the output
            print(f"  {Path(src).relative_to(WORK_path)}: {why}")
        if len(failed) > 20:
            print(f"  ... and {len(failed) - 20} more")
        print(f"Everything else was saved ({counts['copied']} copied, {counts['unchanged']} unchanged).")
        print("Drive still has the previous version of each failed file, if it had one.")
        print("Fix the cause (see the table above this cell) and re-run: only what's still")
        print("missing or changed is copied. The save marker was NOT updated.")
        # Stop with a short error: the cell turns red and "Run all" stops here.
        # 'from None' hides the original exception's huge list.
        raise RuntimeError(f"Save incomplete: {len(failed)} item(s) failed") from None

    # Stamp the marker only after a complete save.
    stamp_marker(WORK_COPY_MARKER_path)
    print(f"Saved {WORK_path} -> {PROJECT_path} in {time.time() - start:.0f} s: "
          f"{counts['copied']} file(s) copied, {counts['unchanged']} unchanged.")
    print(f"  Last loaded from Drive:  {read_marker(Path(SETTINGS['DRIVE_COPY_MARKER']))}")
    print(f"  Last saved to Drive:     {read_marker(WORK_COPY_MARKER_path)}")

Saved /content/my_project -> /content/drive/MyDrive/Colab Notebooks/Project Storage/Bioinformatics Portfolio in 58 s: 164 file(s) copied, 110 unchanged.
  Last loaded from Drive:  2026-11-22 22:34 UTC, 588 min ago
  Last saved to Drive:     2026-11-23 08:22 UTC, 0 min ago


# Two Sample Workflow: Build Simulation Files

## SETUP: New Settings, check files

In [21]:
import os
import shutil
from pathlib import Path

# Needs notebook 01's sections 1-9 run first in this session: SETTINGS, the
# PATH change (section 7) and BAMSurgeon (section 8) all come from there.
if "SETTINGS" not in globals():
    raise RuntimeError("SETTINGS is missing: run notebook 01 sections 1-9 first.")

# Keep BAMs on Drive? False (default) puts them in local scratch, which section
# 11 never saves: they are about 1 GB each and can be rebuilt from the FASTQs.
# True puts them under WORK, so section 11 saves them. Check Drive space first.
KEEP_BAMS_ON_DRIVE = False

WORK = SETTINGS["WORK"]
NB2 = {
    # Local-only space for big temporary files. It sits outside WORK, so it is
    # never saved to Drive, and it disappears when the session ends.
    "SCRATCH": "/content/scratch",
    "REF_DIR": f"{WORK}/reference/genome",          # masked GRCh38 and its indexes (step 0)
    "DESIGN_DIR": f"{WORK}/reference/sureselect",   # Agilent files, cleaned BEDs (step 1)
    "TRUTH_DIR": f"{WORK}/data/raw/truth",          # spike-in list, truth VCFs (steps 2, 6)
    "SIM_DIR": f"{WORK}/data/raw/sim",              # read-source regions, FASTQs (steps 3, 4, 7)
    "CALLS_DIR": f"{WORK}/data/processed/calls",    # the test pipeline's VCFs (step 8)
    "QC_DIR": f"{WORK}/output/results/qc",          # depth, capture, spike-in reports
    "EVAL_DIR": f"{WORK}/output/results/eval",      # scoring and the results table (step 9)
}
NB2["BAM_DIR"] = f"{WORK}/data/processed/bam" if KEEP_BAMS_ON_DRIVE else f"{NB2['SCRATCH']}/bam"

# Files that several steps use.
NB2["REF"] = f"{NB2['REF_DIR']}/hg38_masked.fa"           # the reference FASTA
NB2["SDF"] = f"{NB2['REF_DIR']}/hg38_masked.sdf"          # RTG's copy of it (a folder)
NB2["GENOME"] = f"{NB2['REF_DIR']}/hg38.genome"           # contig lengths, for bedtools
NB2["HAEM_DIR"] = f"{NB2['DESIGN_DIR']}/Haem_1"           # the unzipped Agilent design
NB2["TARGETS_BED"] = f"{NB2['DESIGN_DIR']}/Regions_SNV.clean.bed"
NB2["BAITS_BED"] = f"{NB2['DESIGN_DIR']}/Covered_SNV.clean.bed"
NB2["TARGETS_IL"] = f"{NB2['DESIGN_DIR']}/targets.interval_list"
NB2["BAITS_IL"] = f"{NB2['DESIGN_DIR']}/baits.interval_list"

# CPUs this session has (2 on free Colab). Every multithreaded command uses
# this, so nothing asks for more threads than the machine has.
NB2["THREADS"] = str(os.cpu_count() or 2)
# Java memory for GATK and Picard: 4 GB of the ~12.7 GB, leaving room others.
# MarkDuplicates would probably use most but 1.2Mb + 8M reads should fit.
NB2["JAVA_MEM"] = "-Xmx4g"

# Keys the step cells read as $NAME. Environment Setup's setting are listed too,
# so it means issues are caught here, not hours later in a step.
REQUIRED_NB2 = [
    # From Environment Setup section
    "PROJECT", "WORK", "LOGS_DIR", "BAMSURGEON_DIR",
    # Folders
    "SCRATCH", "REF_DIR", "DESIGN_DIR", "TRUTH_DIR", "SIM_DIR",
    "CALLS_DIR", "QC_DIR", "EVAL_DIR", "BAM_DIR",
    # Files several steps use
    "REF", "SDF", "GENOME", "HAEM_DIR", "TARGETS_BED", "BAITS_BED",
    "TARGETS_IL", "BAITS_IL",
    # Resources
    "THREADS", "JAVA_MEM",
]
missing = [key for key in REQUIRED_NB2 if key not in os.environ]
if missing:
    raise RuntimeError(f"Missing from the environment: {missing}. "
                       "Re-run notebook 01 section 1, then this cell.")

# Export for the %%bash cells, then make every folder. exist_ok=True: no error
# if a folder is already there (for example, copied down from Drive).
os.environ.update(NB2)
for key in ("SCRATCH", "REF_DIR", "DESIGN_DIR", "TRUTH_DIR", "SIM_DIR",
            "CALLS_DIR", "QC_DIR", "EVAL_DIR", "BAM_DIR"):
    Path(NB2[key]).mkdir(parents=True, exist_ok=True)

# Check every tool this notebook calls can be found. shutil.which searches
# PATH the way the shell does, so this also confirms section 7 ran.
tools = ["samtools", "bcftools", "bedtools", "bgzip", "tabix", "bwa", "art_illumina",
         "mosdepth", "picard", "gatk", "rtg", "bamsurgeon", "Rscript", "curl"]
missing = [t for t in tools if shutil.which(t) is None]
if missing:
    raise RuntimeError(f"Not found on PATH: {missing}. Run notebook 01 sections 6-8 first.")

for key in NB2:
    print(f"{key:12} {os.environ.get(key, 'NOT SET')}")
print("All tools found.")

SCRATCH      /content/scratch
REF_DIR      /content/my_project/reference/genome
DESIGN_DIR   /content/my_project/reference/sureselect
TRUTH_DIR    /content/my_project/data/raw/truth
SIM_DIR      /content/my_project/data/raw/sim
CALLS_DIR    /content/my_project/data/processed/calls
QC_DIR       /content/my_project/output/results/qc
EVAL_DIR     /content/my_project/output/results/eval
BAM_DIR      /content/scratch/bam
REF          /content/my_project/reference/genome/hg38_masked.fa
SDF          /content/my_project/reference/genome/hg38_masked.sdf
GENOME       /content/my_project/reference/genome/hg38.genome
HAEM_DIR     /content/my_project/reference/sureselect/Haem_1
TARGETS_BED  /content/my_project/reference/sureselect/Regions_SNV.clean.bed
BAITS_BED    /content/my_project/reference/sureselect/Covered_SNV.clean.bed
TARGETS_IL   /content/my_project/reference/sureselect/targets.interval_list
BAITS_IL     /content/my_project/reference/sureselect/baits.interval_list
THREADS      2
JAVA_MEM 

## STEP 0: Get Index and Masked References

Using GIAB's masked GRCh38 (no alt contigs, and GRC's false duplication fix for chr21 copy of U2AF1). Unpack FASTA and generate .fai index.

Note: GIAB's masked GRCh38: NCBI's no-alt analysis set with the Genome Reference Consortium's false duplications replaced by Ns, including the chr21 copy of U2AF1 that Miller et al. described. Coordinates don't change, and GIAB maps its own data to this family of references, which keeps the later HG002 check consistent.
*   In: GIAB's GCA_000001405.15_GRCh38_no_alt_analysis_set_maskedGRC_exclusions_v2.fasta.gz, downloaded into scratch.
*   Out: reference/genome/hg38_masked.fa with its .fai, .dict, bwa index and RTG SDF, plus hg38.genome and the step0.done marker.
*   Then save: recommended to toggle the Save switch (Save Work cell) and run straight away. Losing this to a disconnect is costly.

In [24]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to build the reference again from scratch.
# Again, note no TRUE boolean in bash, this is just a string.
# Note: this will still build ref first time even if REDO = false.
REDO=false

# step0.done is written last, so it only exists after a complete build. Section
# 2 copies it down from Drive with the reference, so later sessions skip this.
DONE="$REF_DIR/step0.done"
if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$REF.bwt" ] && [ -d "$SDF" ]; then
  echo "Reference and indexes already in $REF_DIR: skipped."
  exit 0
fi
rm -f "$DONE"
SECONDS=0   # bash's built-in timer

# Drive needs about 10 GB free for the result (saved in section 11).
# -B1G reports in whole GB; --output=avail prints only the free space.
FREE_GB=$(df -B1G --output=avail /content/drive | tail -n 1 | tr -d ' ')
echo "Drive free space: ${FREE_GB} GB (the reference needs about 10 GB)."
if [ "$FREE_GB" -lt 11 ]; then
  echo "WARNING: free up Drive space before saving this step in section 11."
fi

# Download GIAB's masked GRCh38 (about 0.9 GB) into scratch: only the
# unpacked FASTA is kept. Why this file:
#   no_alt_analysis_set      primary chromosomes only, so reads don't get two
#                            equally good homes on alternate-haplotype contigs.
#   maskedGRC_exclusions_v2  regions the GRC flagged as false duplications are
#                            Ns, so reads from the real U2AF1 (chr21q) map uniquely.
#   chr-prefixed names       chr1, chr2 ... match the Agilent BED files.
# curl: -f fail on an HTTP error (so set -e stops the cell), -L follow
# redirects, --retry 3 retry a dropped connection, -o output file.
FA_GZ="$SCRATCH/GCA_000001405.15_GRCh38_no_alt_analysis_set_maskedGRC_exclusions_v2.fasta.gz"
curl -fL --retry 3 -o "$FA_GZ" \
  "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/references/GRCh38/$(basename "$FA_GZ")"

# Unpack to plain FASTA (support for compressed references varies between
# tools), then delete the download to free scratch space.
gunzip -c "$FA_GZ" > "$REF"
rm -f "$FA_GZ"

# .fai index: lets tools jump to any region without reading the whole 3 GB.
samtools faidx "$REF"

# Two-column genome file (contig, length): bedtools uses it in step 3 so
# padding stops at chromosome ends.
cut -f1,2 "$REF.fai" > "$GENOME"

# .dict sequence dictionary: GATK and Picard refuse a reference without one.
# ${REF%.fa}.dict means "REF with .fa cut off the end, plus .dict". GATK won't
# overwrite an old one, so remove it first.
rm -f "${REF%.fa}.dict"
gatk --java-options "$JAVA_MEM" CreateSequenceDictionary -R "$REF"

# bwa's search index: the slow part, about an hour (estimate) and about 5 GB
# RAM, single-threaded. Its progress messages go to a log file instead of
# flooding the cell.
echo "Building the bwa index (about an hour)..."
bwa index "$REF" 2> "$LOGS_DIR/step0_bwa_index.log"

# RTG's own copy of the reference (an SDF folder). vcfeval (step 9) compares
# variants by the sequence they produce, and needs it. rtg refuses an
# existing output folder, so remove any old one first.
rm -rf "$SDF"
rtg format -o "$SDF" "$REF" > "$LOGS_DIR/step0_rtg_format.log"

touch "$DONE"
echo "Done in $((SECONDS / 60)) min."

# Check the U2AF1 fix: the false copy (chr21 6.5 Mb) must be all N, and the
# real gene (chr21 43.09 Mb) must have bases.
echo "False copy: $(samtools faidx "$REF" chr21:6500001-6500060 | tail -n +2 | tr -d '\n')"
echo "Real U2AF1: $(samtools faidx "$REF" chr21:43094501-43094560 | tail -n +2 | tr -d '\n')"

Reference and indexes already in /content/my_project/reference/genome: skipped.


## STEP 1: Clean design files + Make ILs

Strip Agilent's track lines with `grep`, keep four columns, drop the nine false U2AF1 rows below chr21:7 Mb in each file, and make Picard interval lists. The cell ends by checking the counts, which were confirmed on your design files here.
*   In: `reference/sureselect/Haem_1/Haem_1_Regions_SNV.bed` and Haem_1_Covered_SNV.bed, plus the .dict from step 0.
*   Out: `Regions_SNV.clean.bed` (targets), `Covered_SNV.clean.bed` (baits), `targets.interval_list` and `baits.interval_list, all` in reference/sureselect/.

In [25]:
%%bash
set -euo pipefail

# Inputs this step needs. [ -s FILE ] is true when FILE exists and isn't empty.
for f in Regions_SNV Covered_SNV; do
  [ -s "$HAEM_DIR/Haem_1_${f}.bed" ] || {
    echo "Missing $HAEM_DIR/Haem_1_${f}.bed."
    exit 1; }
done
[ -s "${REF%.fa}.dict" ] || { echo "No reference dictionary: run step 0 first."; exit 1; }

# Clean the two design files we need:
#   Regions_SNV = exon targets    -> where variants are called and scored
#   Covered_SNV = probe footprint -> where reads come from (step 3)
for f in Regions_SNV Covered_SNV; do
  # 1) Drop Agilent's 'track' and 'browser' header lines; most tools reject
  #    them. -v keeps lines that DON'T match; -E lets (a|b) mean either word.
  grep -v -E '^(track|browser)' "$HAEM_DIR/Haem_1_${f}.bed" |
  # 2) Tab-separated output (OFS), skipping chr21 rows that start below 7 Mb:
  #    the nine rows on the false U2AF1 copy, which is all N in the masked
  #    reference. Keep chromosome, start, end and gene name only.
  #    awk: OFS="\t" sets to tsv, "!()" flips $1== and $2< filter, so that
  #    only Chr21 <7Mb (false U2AF1) are caught.
  awk 'BEGIN{OFS="\t"} !($1=="chr21" && $2<7000000) {print $1,$2,$3,$4}' |
  # 3) Sort by chromosome, then numerically (-n) by start.
  #    sort: -k1,1 (from field1:field1) -k2,2n (from field2:field2 as number)
  sort -k1,1 -k2,2n > "$DESIGN_DIR/${f}.clean.bed"
done

# Picard takes "interval lists": the same regions plus a header copied from the
# reference's .dict (-SD), so contig names and order match the BAMs. Picard uses
# the header to confirm the regions use the same contig names and order as the BAM,
# since both come from the same reference. Interval lists needed for CollectHsMetrics.
# Command accepts Java memory options such as -Xmx4g before the tool name.
picard "$JAVA_MEM" BedToIntervalList -I "$TARGETS_BED" -O "$TARGETS_IL" -SD "${REF%.fa}.dict"
picard "$JAVA_MEM" BedToIntervalList -I "$BAITS_BED" -O "$BAITS_IL" -SD "${REF%.fa}.dict"

# Check against the counts this design gives.
check() {   # check FILE EXPECTED_ROWS EXPECTED_BP
  local rows bp
  rows=$(wc -l < "$1")
  bp=$(awk '{s += $3 - $2} END {print s}' "$1")
  if [ "$rows" -eq "$2" ] && [ "$bp" -eq "$3" ]; then
    echo "OK        $(basename "$1"): $rows rows, $bp bp"
  else
    echo "MISMATCH  $(basename "$1"): $rows rows, $bp bp (expected $2 rows, $3 bp)"
    return 1
  fi
}
check "$TARGETS_BED" 2170 380700
check "$BAITS_BED" 2068 485357

OK        Regions_SNV.clean.bed: 2170 rows, 380700 bp
OK        Covered_SNV.clean.bed: 2068 rows, 485357 bp


00:36:26.434 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/micromamba/envs/bioinf_portfolio/share/picard-3.5.0-1/picard.jar!/com/intel/gkl/native/libgkl_compression.so
[Mon Nov 23 00:36:26 UTC 2026] BedToIntervalList --INPUT /content/my_project/reference/sureselect/Regions_SNV.clean.bed --SEQUENCE_DICTIONARY /content/my_project/reference/genome/hg38_masked.dict --OUTPUT /content/my_project/reference/sureselect/targets.interval_list --SORT true --UNIQUE false --DROP_MISSING_CONTIGS false --KEEP_LENGTH_ZERO_INTERVALS false --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 5 --MAX_RECORDS_IN_RAM 500000 --CREATE_INDEX false --CREATE_MD5_FILE false --help false --version false --showHidden false --USE_JDK_DEFLATER false --USE_JDK_INFLATER false
[Mon Nov 23 00:36:26 UTC 2026] Executing as root@960882c89fb7 on Linux 6.6.159+ amd64; OpenJDK 64-Bit Server VM 17.0.18-internal+0-adhoc.rattler.src; Deflater: Intel; Inflater: Intel;

## STEP 2: Build the truth-variant VCF

Turn the truth-set table into checked hg38 records, each with its target VAF. Two cells: a Python cell checks a fixed table of positions offline, then a bash cell writes and checks the VCF.
*   In: the truth-set table (fixed in the first cell), the cleaned targets, the reference.
*   Out: data/raw/truth/variants.tsv (working list), truth_set.tsv (the same rows with MANE transcript, HGVS and IDs, for the write-up), spike.vcf (BAMSurgeon's input) and variants.norm.tsv (the normalised list step 9 joins on).

The first cell holds the truth set as a fixed table of GRCh38 positions. Each was looked up on GeneBe on 20 November 2026, which confirmed its HGVS on the MANE Select transcript. An earlier version asked Ensembl's REST API at run time, but Ensembl blocked Colab's shared network address. A fixed table also means a rerun can never quietly change the truth set. "Why a fixed table, and where the positions go", at the end of this step, explains the lookup and what the later steps do with each position. Offline, the cell checks that every row passes three tests:
*   the position lies inside a panel target, which also catches one on the false U2AF1 copy;
*   the reference carries the REF bases there, and each duplication inserts a copy of the bases right after its anchor;
*   it is at least 300 bp from every other spiked variant.
It stops with a list of problems if any check fails. It needs no internet; samtools is found through section 7's PATH.

In [26]:
import os
import subprocess
from pathlib import Path

TRUTH_DIR = Path(os.environ["TRUTH_DIR"])
REF = os.environ["REF"]

# The truth set, as a fixed table (see "The truth set"). Each row is one GRCh38
# position written the VCF way: 1-based, and an indel starts at the base just
# before it (the anchor), shifted as far left as it can go. Every row was
# checked on GeneBe (genebe.net/variant/hg38/CHROM-POS-REF-ALT) on 20 Nov 2026,
# which reports the HGVS on the MANE Select transcript given here. The cell
# re-checks what it can offline: REF bases, targets, duplications, spacing.
# Two rows use MANE numbering that differs from the familiar names:
#   MYD88 "L265P" is p.Leu252Pro (c.755T>C) on NM_002468.5.
#   CALR type 1, often written c.1092_1143del, is c.1099_1150del: the same
#   52 bp deletion, described from its rightmost position as HGVS requires.
CALR_DEL52 = "GCAGAGGCTTAAGGAGGAGGAAGAAGACAAGAAACGCAAAGAGGAGGAGGAG"   # the deleted bases
TRUTH = [
    # name            VAF   chrom    position   REF                ALT      MANE Select       HGVS c.           HGVS p.               ID
    ("JAK2_V617F",     0.40, "chr9",  5073770,   "G",               "T",     "NM_004972.4",    "c.1849G>T",      "p.Val617Phe",        "rs77375493"),
    ("SF3B1_K700E",    0.40, "chr2",  197402110, "T",               "C",     "NM_012433.4",    "c.2098A>G",      "p.Lys700Glu",        "rs559063155"),
    ("TP53_R273H",     0.40, "chr17", 7673802,   "C",               "T",     "NM_000546.6",    "c.818G>A",       "p.Arg273His",        "rs28934576"),
    ("ASXL1_G646Wfs12", 0.40, "chr20", 32434638, "A",               "AG",    "NM_015338.6",    "c.1934dup",      "p.Gly646TrpfsTer12", "ClinVar:426927"),
    ("IDH1_R132H",     0.10, "chr2",  208248388, "C",               "T",     "NM_005896.4",    "c.395G>A",       "p.Arg132His",        "rs121913500"),
    ("SRSF2_P95H",     0.10, "chr17", 76736877,  "G",               "T",     "NM_001195427.2", "c.284C>A",       "p.Pro95His",         "rs751713049"),
    ("MYD88_L265P",    0.10, "chr3",  38141150,  "T",               "C",     "NM_002468.5",    "c.755T>C",       "p.Leu252Pro",        "rs387907272"),
    ("CALR_type1",     0.10, "chr19", 12943750,  "A" + CALR_DEL52,  "A",     "NM_004343.4",    "c.1099_1150del", "p.Leu367ThrfsTer46", "rs1555760738"),
    ("IDH2_R140Q",     0.05, "chr15", 90088702,  "C",               "T",     "NM_002168.4",    "c.419G>A",       "p.Arg140Gln",        "rs121913502"),
    ("BRAF_V600E",     0.05, "chr7",  140753336, "A",               "T",     "NM_004333.6",    "c.1799T>A",      "p.Val600Glu",        "rs113488022"),
    ("U2AF1_S34F",     0.05, "chr21", 43104346,  "G",               "A",     "NM_006758.3",    "c.101C>T",       "p.Ser34Phe",         "rs371769427"),
    ("NRAS_G12D",      0.02, "chr1",  114716126, "C",               "T",     "NM_002524.5",    "c.35G>A",        "p.Gly12Asp",         "rs121913237"),
    ("KRAS_G12D",      0.02, "chr12", 25245350,  "C",               "T",     "NM_004985.5",    "c.35G>A",        "p.Gly12Asp",         "rs121913529"),
    ("DNMT3A_R882H",   0.02, "chr2",  25234373,  "C",               "T",     "NM_022552.5",    "c.2645G>A",      "p.Arg882His",        "rs147001633"),
    ("RHOA_G17V",      0.02, "chr3",  49375540,  "C",               "A",     "NM_001664.4",    "c.50G>T",        "p.Gly17Val",         "rs11552761"),
    ("NPM1_typeA",     0.02, "chr5",  171410539, "C",               "CTCTG", "NM_002520.7",    "c.860_863dup",   "p.Trp288CysfsTer12", "rs587776806"),
    ("DDX41_D140Gfs2", 0.50, "chr5",  177515944, "T",               "TCATC", "NM_016222.4",    "c.415_418dup",   "p.Asp140GlyfsTer2",  "rs762890562"),
    ("TP53_P72R",      0.50, "chr17", 7676154,   "G",               "C",     "NM_000546.6",    "c.215C>G",       "p.Pro72Arg",         "rs1042522"),
    ("TET2_I1762V",    0.50, "chr4",  105275794, "A",               "G",     "NM_001127208.3", "c.5284A>G",      "p.Ile1762Val",       "rs2454206"),
    ("FLT3_T227M",     0.50, "chr13", 28050157,  "G",               "A",     "NM_004119.3",    "c.680C>T",       "p.Thr227Met",        "rs1933437"),
]


def ref_seq(chrom, start, end):
    """Bases chrom:start-end (1-based, inclusive) from the masked reference."""
    out = subprocess.run(["samtools", "faidx", REF, f"{chrom}:{start}-{end}"],
                         check=True, capture_output=True, text=True).stdout
    return "".join(out.splitlines()[1:]).upper()   # drop the >name line, join the rest


# Target regions, to confirm every position is on the panel. This also catches
# a position on the false U2AF1 copy, whose rows step 1 removed.
TARGETS = {}
for line in open(os.environ["TARGETS_BED"]):
    chrom, start, end = line.split("\t")[:3]
    TARGETS.setdefault(chrom, []).append((int(start), int(end)))


def in_targets(chrom, pos):
    """True if the 1-based position lies in a target (BED starts are 0-based)."""
    return any(start < pos <= end for start, end in TARGETS.get(chrom, []))


rows, problems = [], []
for name, vaf, chrom, pos, ref, alt, tx, hgvs_c, hgvs_p, vid in TRUTH:
    # 1) The reference must carry the REF bases at this position. A wrong
    #    position, or a typo in REF, fails here.
    found = ref_seq(chrom, pos, pos + len(ref) - 1)
    if found != ref:
        problems.append(f"{name}: reference has {found[:12]!r} at {chrom}:{pos}, table says {ref[:12]!r}")
    # 2) The position must be inside a panel target.
    if not in_targets(chrom, pos):
        problems.append(f"{name}: {chrom}:{pos} is not inside a target")
    # 3) Indels keep the anchor base. A duplication, shifted left, inserts a copy
    #    of the bases straight after the anchor, so those must match.
    if len(alt) > len(ref):
        inserted = alt[len(ref):]
        after = ref_seq(chrom, pos + 1, pos + len(inserted))
        if alt[0] != ref[0] or inserted != after:
            problems.append(f"{name}: inserts {inserted!r}, but the bases after the anchor are {after!r}")
    elif len(ref) > len(alt) and alt != ref[0]:
        problems.append(f"{name}: a deletion's ALT must be the anchor base alone")
    rows.append((chrom, pos, ref, alt, vaf, name, tx, hgvs_c, hgvs_p, vid))
    shown = f"{len(ref) - 1} bp deleted" if len(ref) > 12 else f"{ref}>{alt}"   # CALR's REF is too long to print
    print(f"{name:16} {chrom + ':' + str(pos):16} {shown:14} {tx}:{hgvs_c} ({hgvs_p})")

# The FLT3-ITD, built by hand: no single ITD is standard. Take 30 bp inside the
# FLT3 exon 14 target (chr13:28,034,082-28,034,214) and insert a copy straight
# after it. 30 bp = 10 codons, so it stays in frame, as real ITDs do. In VCF an
# insertion is written at the base before the new bases (the anchor): here the
# last base of the stretch, with ALT = anchor + the 30 bp.
dup = ref_seq("chr13", 28034101, 28034130)
if len(dup) != 30 or "N" in dup:
    problems.append(f"FLT3_ITD30: unexpected reference sequence {dup!r}")
else:
    rows.append(("chr13", 28034130, dup[-1], dup[-1] + dup, 0.05, "FLT3_ITD30",
                 "NM_004119.3", "30 bp ITD built from chr13:28034101-28034130", ".", "."))
    print(f"{'FLT3_ITD30':16} {'chr13:28034130':16} {'+30 bp':14} copy of chr13:28034101-28034130 (exon 14)")

# Spacing: keep spiked variants at least ~300 bp apart (one fragment length),
# so no read pair carries two of them.
rows.sort(key=lambda r: (r[0], r[1]))
for a, b in zip(rows, rows[1:]):
    if a[0] == b[0] and b[1] - a[1] < 300:
        problems.append(f"{a[5]} and {b[5]} are only {b[1] - a[1]} bp apart (need >= 300)")

if problems or len(rows) != 21:
    raise RuntimeError(f"{len(rows)} rows, {len(problems)} problem(s):\n  " + "\n  ".join(problems))

# Working list for step 2's bash cell: CHROM POS REF ALT VAF NAME, no header.
with open(TRUTH_DIR / "variants.tsv", "w") as out:
    for r in rows:
        out.write("\t".join(str(x) for x in r[:6]) + "\n")
# The full table with its annotation, for the README and the results write-up.
with open(TRUTH_DIR / "truth_set.tsv", "w") as out:
    out.write("chrom\tpos\tref\talt\tvaf\tname\tmane_select\thgvs_c\thgvs_p\tid\n")
    for r in rows:
        out.write("\t".join(str(x) for x in r) + "\n")
print(f"\nAll 21 variants passed the checks.")
print(f"Wrote {TRUTH_DIR / 'variants.tsv'} and {TRUTH_DIR / 'truth_set.tsv'}")

JAK2_V617F       chr9:5073770     G>T            NM_004972.4:c.1849G>T (p.Val617Phe)
SF3B1_K700E      chr2:197402110   T>C            NM_012433.4:c.2098A>G (p.Lys700Glu)
TP53_R273H       chr17:7673802    C>T            NM_000546.6:c.818G>A (p.Arg273His)
ASXL1_G646Wfs12  chr20:32434638   A>AG           NM_015338.6:c.1934dup (p.Gly646TrpfsTer12)
IDH1_R132H       chr2:208248388   C>T            NM_005896.4:c.395G>A (p.Arg132His)
SRSF2_P95H       chr17:76736877   G>T            NM_001195427.2:c.284C>A (p.Pro95His)
MYD88_L265P      chr3:38141150    T>C            NM_002468.5:c.755T>C (p.Leu252Pro)
CALR_type1       chr19:12943750   52 bp deleted  NM_004343.4:c.1099_1150del (p.Leu367ThrfsTer46)
IDH2_R140Q       chr15:90088702   C>T            NM_002168.4:c.419G>A (p.Arg140Gln)
BRAF_V600E       chr7:140753336   A>T            NM_004333.6:c.1799T>A (p.Val600Glu)
U2AF1_S34F       chr21:43104346   G>A            NM_006758.3:c.101C>T (p.Ser34Phe)
NRAS_G12D        chr1:114716126   C>T            NM

Write the VCF and check every REF base against the reference:

In [27]:
%%bash
set -euo pipefail

[ -s "$TRUTH_DIR/variants.tsv" ] || { echo "No variants.tsv: run the cell above first."; exit 1; }

# Build a minimal VCF: header lines, then one line per variant.
{
  printf '##fileformat=VCFv4.2\n'
  # The two INFO fields: BAMSurgeon reads VAF as the fraction to spike; NAME
  # is only a label for us.
  printf '##INFO=<ID=VAF,Number=1,Type=Float,Description="Target allele fraction">\n'
  printf '##INFO=<ID=NAME,Number=1,Type=String,Description="Variant name">\n'
  # One ##contig line per reference sequence, so bcftools knows the order.
  awk '{printf "##contig=<ID=%s,length=%s>\n", $1, $2}' "$GENOME"
  printf '#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n'
  # ID and QUAL "." (empty), FILTER PASS, INFO from columns 5 and 6.
  awk 'BEGIN{OFS="\t"} {print $1,$2,".",$3,$4,".","PASS","VAF="$5";NAME="$6}' "$TRUTH_DIR/variants.tsv"
} > "$TRUTH_DIR/spike.unsorted.vcf"

# Sort into reference order (-Ov = plain-text VCF).
bcftools sort "$TRUTH_DIR/spike.unsorted.vcf" -Ov -o "$TRUTH_DIR/spike.sorted.vcf"

# Normalise, which also catches mistakes:
#   -f REF         compare against the reference
#   --check-ref e  stop with an error if any REF base doesn't match it
#   -m -both       split any multi-allelic record into one per ALT
#   (always)       left-align indels, so each has one standard position
bcftools norm -f "$REF" --check-ref e -m -both "$TRUTH_DIR/spike.sorted.vcf" -Ov -o "$TRUTH_DIR/spike.vcf"

# The same list as spike.vcf after normalisation. Step 9 joins results on
# these positions, which is what the truth VCF will carry.
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\t%INFO/NAME\n' "$TRUTH_DIR/spike.vcf" \
  > "$TRUTH_DIR/variants.norm.tsv"

echo "Records in spike.vcf: $(grep -vc '^#' "$TRUTH_DIR/spike.vcf") (expect 21)"

Records in spike.vcf: 21 (expect 21)


Writing to /tmp/bcftools.leHddK
Merging 1 temporary files
Done
Cleaning
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	21/0/0/1/0/0/0


Check: 21 records, and bcftools norm finished without a REF-mismatch error. Its summary line shows realigned 1: the FLT3-ITD, which step 2 anchors at the right end of the duplicated stretch and bcftools shifts to the left end, the standard VCF form. The other indels were already left-aligned (seen on the first Colab run).

What the first run showed (20 Nov 2026). All 21 rows passed the Python cell's checks, spike.vcf has 21 records, and bcftools norm realigned exactly one. Three results show that the table and your reference agree:

*   CALR type 1: all 53 REF bases matched. The row's REF is the anchor base plus the 52 deleted bases, and the cell compares every one with the reference. At a wrong position, or with the deletion written from the wrong end, the bases would stop matching within the first few, and 53 in a row don't match by chance. It also confirms the two ways of writing this deletion describe the same 52 bases: VCF from its left end (chr19:12,943,750), HGVS from its right end (c.1099_1150del).
*  U2AF1 S34F: the REF base G was found at chr21:43,104,346. That is the real U2AF1 gene, which the masked reference keeps; only the false copy near chr21:6.5 Mb is turned to N (step 0's printout showed both). Had the masking hit the real gene, the cell would have read N there and stopped. So reads from U2AF1 have one place to map, and this variant can be spiked and called.
*   FLT3-ITD: moved from chr13:28,034,130 to 28,034,100, exactly 30 bases. Step 2 writes the inserted copy at the right end of the duplicated stretch, but VCF places an insertion at the leftmost spot it could occupy. An exact tandem copy can slide back by its own length, so the anchor moved to the base just before the stretch: 28,034,130 − 30 = 28,034,100. It stopped there because that base differs from the stretch's last base, so the repeat goes no further. A shift of exactly the duplication's length is what a correctly built tandem duplication should do, and the new position is still inside the exon 14 target (28,034,082–28,034,214). Check it with grep FLT3_ITD30 "$TRUTH_DIR/variants.norm.tsv", which shows chr13  28034100. Step 9 joins on this normalised position, which is the one BAMSurgeon's truth file will carry.

The other three indels (ASXL1, NPM1, DDX41) weren't moved, so GeneBe's positions were already left-aligned. Their duplication check also passed: each inserts a copy of the bases right after its anchor, as a duplication should.

### Why a fixed table, and where the positions go

What has to be translated. The truth-set table describes each variant the way a clinical report does: a gene, and an HGVS change on one of its transcripts, such as JAK2 c.1849G>T (p.Val617Phe). Every tool after step 2 works in genome coordinates instead: chromosome, position, reference base(s) and alternative base(s). So something has to turn "c.1849G>T in JAK2" into "chr9, position 5,073,770, G becomes T".

Why the transcript matters. c.1849 means the 1,849th base of the coding sequence, counted from the A of the start codon along one particular transcript, with the introns skipped. A gene's transcripts can differ in their exons or start codon, so the same c. number can sit on different genomic bases. MYD88 shows this: the base called c.794 on the older transcript NM_002468.4 is c.755 on the MANE transcript NM_002468.5, which has 39 fewer coding bases before it. Clinical reports use MANE Select, so the table does too.

What the Ensembl version did, for each row (JAK2 V617F as the example):
1.  Find the transcript. Ask Ensembl's lookup endpoint for JAK2's transcripts and keep the one flagged canonical. For genes with a MANE Select transcript, Ensembl makes that the canonical one: here ENST00000381652, which matches RefSeq's NM_004972.4.
2.  Build the full HGVS. Join the two: ENST00000381652:c.1849G>T.
3.  Convert it. Send that to Ensembl's Variant Recoder. Ensembl stores where each exon of the transcript lies on GRCh38, so it can walk along the coding exons to base 1,849 and report the genomic position in VCF form, 9-5073770-G-T, plus the protein change it causes, p.Val617Phe. For a gene on the minus strand the bases come back complemented: TP53 c.818G>A is C>T on the genome.
4.  Check it. Keep the position only if it lies inside a panel target, the protein change matches the table, and it is at least 300 bp from every other variant.
5.  Write it down. Add one line to variants.tsv: chr9  5073770  G  T  0.40  JAK2_V617F.
The transcript is used only in steps 1–3. From step 4 on, every file holds just chromosome, position, REF, ALT, VAF and name.
What the fixed table changes. Steps 1–3 were done once, by hand, on GeneBe, which shows the HGVS on the MANE Select transcript for any GRCh38 position, and their results are typed into the cell. The cell starts at step 4, with one difference: without the transcript's exon map it can't work out the protein change, so it checks the REF bases against the reference instead, alongside the target, duplication and spacing checks. Step 5 is unchanged, so every later step gets exactly the same file either way.
Where the JAK2 position goes next.
1.  Step 2, bash cell. variants.tsv becomes a VCF record in spike.vcf: chr9  5073770  .  G  T  .  PASS  VAF=0.4;NAME=JAK2_V617F. bcftools norm confirms the reference really has a G at chr9:5,073,770 and shifts any indel to its leftmost position. The normalised list is saved as variants.norm.tsv.
2.  Step 6, BAMSurgeon. It reads that record and collects Sample A's reads covering chr9:5,073,770 (roughly 900). It picks 40% of them at random, changes the G to T in those reads and realigns them. The result is Sample B. It also writes truth.vcf.gz, recording what actually went in and the VAF it achieved.
3.  Step 7. Sample B is turned back into FASTQ, so the test pipeline starts from raw reads, as it would with a real sample.
4.  Step 8, the pipeline under test. It aligns the reads and calls variants with Mutect2, without ever seeing the truth set. If it works, B.pass.vcf.gz holds a call at chr9:5,073,770 G>T with an allele fraction near 0.4.
5.  Step 9, scoring. vcfeval lines up truth.vcf.gz against the pipeline's calls. A truth variant with a matching call is a true positive, one with no call is a false negative, and a call that matches nothing is a false positive. The R table then joins vcfeval's verdicts to variants.norm.tsv on chromosome and position, so each row gets its name and tier back: "JAK2_V617F, 40%, found". Sample A had nothing spiked in, so any call inside its targets counts as a false positive.
So the positions are the answer key. They tell BAMSurgeon where to write each variant, and they tell vcfeval where the pipeline should find it.
Why the table is fixed rather than looked up at run time.
*   Ensembl's REST service refused Colab's shared network address on 20 November 2026, so the lookup couldn't run there.
*   A truth set should be a fixed, versioned file. A live lookup follows Ensembl's current release, so a rerun could quietly pick a different transcript.
*   The live version needed each HGVS to be exactly right on the current MANE transcript. The plan's MYD88 row, c.794T>C, would have failed its protein check.
*   What's lost: the protein check was done once, on GeneBe (see Sources), rather than on every run.

## STEP 3: Build the read-source regions

Pad the bait footprint by 200 bp so coverage tapers off past the probe edges, as in real capture, then merge and extract the sequence. On your design this gives 1,600 regions and 1,206,514 bp, the numbers the cell checks against (computed here from the cleaned files).
*  In: Covered_SNV.clean.bed, hg38.genome, the reference.
*  Out: data/raw/sim/panel_src.bed and panel_src.fa.

Notes:
*  BAITS_BED - cleaned design files, from

In [28]:
%%bash
set -euo pipefail

for f in "$BAITS_BED" "$GENOME" "$REF"; do
  [ -s "$f" ] || { echo "Missing $f: run steps 0 and 1 first."; exit 1; }
done

# Pad, merge and extract the regions ART will simulate reads from.
#   bedtools slop -b 200  widen every probe region by 200 bp on both sides.
#                         Capture pulls whole DNA fragments (~250 bp here), so
#                         real coverage runs past the probe edges and tapers
#                         off; simulating from padded regions reproduces that.
#   -g GENOME             contig lengths, so padding stops at chromosome ends.
bedtools slop -b 200 -g "$GENOME" -i "$BAITS_BED" |
  # Padded neighbours now overlap: sort, then merge overlaps into one region,
  # so no stretch is simulated twice (which would double its depth).
  sort -k1,1 -k2,2n |
  bedtools merge > "$SIM_DIR/panel_src.bed"

# The DNA sequence of each region, one FASTA entry per region.
bedtools getfasta -fi "$REF" -bed "$SIM_DIR/panel_src.bed" -fo "$SIM_DIR/panel_src.fa"

echo "Regions: $(wc -l < "$SIM_DIR/panel_src.bed") (expect 1600)"
echo "Bases:   $(awk '{s += $3 - $2} END {print s}' "$SIM_DIR/panel_src.bed") (expect 1206514)"

Regions: 1600 (expect 1600)
Bases:   1206514 (expect 1206514)


## STEP 4: Simulate Sample A

ART simulates 2×150 reads from every padded region. The HSXt profile (HiSeq X TruSeq, 150 bp) is the closest built-in to a modern 2×150 run, since ART has no NovaSeq profile. The fixed seed makes the run reproducible. ART's raw output (~2.7 GB) goes to scratch; only the compressed FASTQs are kept.
*   In: data/raw/sim/panel_src.fa.
*   Out: data/raw/sim/A_1.fq.gz and A_2.fq.gz. This is Sample A.
*   Saving files to storage recommended after this step.

**Note:**
*  ART doesn't simulate adapter read-through, so the test pipeline skips adapter trimming. A pipeline for real data needs a trimming step.
*  ART simulates errors in line with HiSeq X, which can give rise to false positives, detected in Test Pipeline stage.

**Check:**
The first run gave 3,490,000 read pairs in 5 minutes. That is about 13% below the simple estimate of 4.0 million (1000 × 1,206,514 bp ÷ 300 bp per pair), but nowhere near double, so -f counts every read, not every pair. Step 5's target depth is the number that matters.

In [29]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to simulate Sample A again.
REDO=false

OUT1="$SIM_DIR/A_1.fq.gz"
OUT2="$SIM_DIR/A_2.fq.gz"
DONE="$SIM_DIR/A.done"   # written last: exists only after a complete run
if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$OUT1" ] && [ -s "$OUT2" ]; then
  echo "Sample A FASTQs already present: skipped."
  exit 0
fi
[ -s "$SIM_DIR/panel_src.fa" ] || { echo "No panel_src.fa: run step 3 first."; exit 1; }
rm -f "$DONE"
SECONDS=0

# art_illumina simulates Illumina reads from every sequence in panel_src.fa.
#   -ss HSXt   built-in error and quality profile: HiSeq X, TruSeq, 150 bp.
#              ART has no NovaSeq profile; this is its closest 2x150 chemistry.
#              The profile sets base qualities and where errors fall along a read.
#   -p         paired-end: one read from each end of every fragment (R1, R2).
#   -l 150     read length, matching 2x150 sequencing.
#   -f 1000    fold coverage: average reads covering each base. 1000x gives
#              about 20 supporting reads for a 2% VAF variant. Step 5 checks it.
#   -m 250     mean fragment (insert) length; capture libraries are usually
#              sheared to ~200-300 bp. With 150 bp reads, the mates of an
#              average fragment overlap by ~50 bp, as in real panel data.
#   -s 50      fragment length standard deviation: most fragments 150-350 bp.
#   -rs 42     random seed: same seed and input give identical reads.
#   -na        no .aln files (large, and their positions are per region, not
#              genomic; step 5 aligns to the whole genome instead).
#   -i, -o     input FASTA; output prefix. Raw FASTQs (~2.7 GB) go to scratch.
# ART doesn't simulate adapter read-through, so the test pipeline skips adapter
# trimming. A pipeline for real data needs a trimming step.
echo "Simulating (single-threaded; about 5 minutes on free Colab)..."
art_illumina -ss HSXt -p -l 150 -f 1000 -m 250 -s 50 -rs 42 -na \
  -i "$SIM_DIR/panel_src.fa" -o "$SCRATCH/A_" > "$LOGS_DIR/step4_art.log"

# Compress into data/raw/sim. bgzip writes gzip-compatible files that every
# tool here reads; -@ uses all CPUs, which plain gzip can't.
bgzip -@ "$THREADS" -c "$SCRATCH/A_1.fq" > "$OUT1"
bgzip -@ "$THREADS" -c "$SCRATCH/A_2.fq" > "$OUT2"
rm -f "$SCRATCH/A_1.fq" "$SCRATCH/A_2.fq"
touch "$DONE"

# Count read pairs: a FASTQ record is 4 lines, so count every 4th line.
PAIRS=$(zcat "$OUT1" | awk 'NR % 4 == 1' | wc -l)
echo "Done in $((SECONDS / 60)) min. Read pairs: $PAIRS"
echo "About 3.5 million expected (3,490,000 on the first run). About 7 million means ART counted pairs:"
echo "rerun with -f 500 and REDO=true."

Sample A FASTQs already present: skipped.


## STEP 5: Align Sample A and confirm the depth

This BAM is BAMSurgeon's input in step 6, so its depth has to be right first. Run steps 5, 6 and 7 in one session since the BAM lives in scratch (not written to Storage)
*   In: Sample A's FASTQs, the reference and its bwa index, the targets and interval lists.
*   Out: A.bam and its index in BAM_DIR; depth and capture metrics in output/results/qc/.

In [30]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to align Sample A again. Takes around 23min.
REDO=false

BAM="$BAM_DIR/A.bam"
DONE="$BAM_DIR/A.done"
if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$BAM" ]; then
  echo "A.bam already present: skipped."
  exit 0
fi
for f in "$SIM_DIR/A_1.fq.gz" "$SIM_DIR/A_2.fq.gz" "$REF.bwt" "$TARGETS_BED" "$TARGETS_IL" "$BAITS_IL"; do
  [ -s "$f" ] || { echo "Missing $f: run steps 0, 1 and 4 first."; exit 1; }
done
rm -f "$DONE"
SECONDS=0

# Align Sample A to the WHOLE genome, so each read competes for its best home.
#   -t THREADS   one thread per CPU (2 on free Colab).
#   -K 100000000 process a fixed number of bases per batch whatever the thread
#                count, which makes the output reproducible. Step 8 relies on it
#                to reuse this alignment for Sample A.
#   -R '@RG...'  read group on every read: ID (this run), SM (sample name; GATK
#                groups reads by it), PL (platform). GATK refuses BAMs without one.
# bwa's progress messages (stderr) go to a log file. Peak memory is about 6 GB.
echo "Aligning Sample A (about 20 minutes on free Colab)..."
bwa mem -t "$THREADS" -K 100000000 -R '@RG\tID:A\tSM:A\tPL:ILLUMINA' \
  "$REF" "$SIM_DIR/A_1.fq.gz" "$SIM_DIR/A_2.fq.gz" 2> "$LOGS_DIR/step5_bwa_A.log" |
  # Sort by position. -@ extra threads; -T where temporary files go.
  samtools sort -@ "$THREADS" -T "$SCRATCH/sortA" -o "$BAM"

# .bai index: lets tools jump to a region instead of reading the whole BAM.
samtools index "$BAM"
touch "$DONE"
echo "Aligned in $((SECONDS / 60)) min."

# Depth over the targets.
#   -t  decompression threads   -n  no per-base output (large, not needed)
#   -b  report depth for each region in this BED; then output prefix, BAM.
# mosdepth counts overlapping mates once, which lowers the depth. But targets
# sit in the middle of each padded region, where coverage is fullest (it
# tapers towards region ends), so the two roughly cancel: the first run gave
# a mean target depth of 1,001 with ART's -f 1000.
mosdepth -t "$THREADS" -n -b "$TARGETS_BED" "$QC_DIR/A" "$BAM"

# Picard capture metrics: on-target rate, depth spread, bases above set depths.
#   -BI baits (probe footprint)  -TI targets (exons)  -R reference
# Expect near-100% on target: no off-target reads were simulated. That is a
# property of the simulation, not a result to report.
picard "$JAVA_MEM" CollectHsMetrics -I "$BAM" -O "$QC_DIR/A.hs_metrics.txt" -R "$REF" \
  -BI "$BAITS_IL" -TI "$TARGETS_IL"

# Mean depth across all targets: column 4 of mosdepth's total_region line.
MEAN=$(awk '$1 == "total_region" {print $4}' "$QC_DIR/A.mosdepth.summary.txt")
echo "Mean target depth: $MEAN (aim for about 800-1,100; must stay under 2,000)"

Aligning Sample A (about 20 minutes on free Colab)...
Aligned in 19 min.
Mean target depth: 1001.41 (aim for about 800-1,100; must stay under 2,000)


[bam_sort_core] merging from 1 files and 2 in-memory blocks...
01:08:33.937 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/micromamba/envs/bioinf_portfolio/share/picard-3.5.0-1/picard.jar!/com/intel/gkl/native/libgkl_compression.so
[Mon Nov 23 01:08:34 UTC 2026] CollectHsMetrics --BAIT_INTERVALS /content/my_project/reference/sureselect/baits.interval_list --TARGET_INTERVALS /content/my_project/reference/sureselect/targets.interval_list --INPUT /content/scratch/bam/A.bam --OUTPUT /content/my_project/output/results/qc/A.hs_metrics.txt --REFERENCE_SEQUENCE /content/my_project/reference/genome/hg38_masked.fa --METRIC_ACCUMULATION_LEVEL ALL_READS --NEAR_DISTANCE 250 --MINIMUM_MAPPING_QUALITY 20 --MINIMUM_BASE_QUALITY 20 --CLIP_OVERLAPPING_READS true --INCLUDE_INDELS false --COVERAGE_CAP 200 --SAMPLE_SIZE 10000 --ALLELE_FRACTION 0.001 --ALLELE_FRACTION 0.005 --ALLELE_FRACTION 0.01 --ALLELE_FRACTION 0.02 --ALLELE_FRACTION 0.05 --ALLELE_FRACTION 0.1 --ALLELE_F

## STEP 6: Spike the variants to make Sample B

Pre-check:

In [31]:
%%bash
awk 'BEGIN{OFS="\t"} {print $1, $2-1, $2}' "$TRUTH_DIR/variants.norm.tsv" > "$SCRATCH/sites.bed"
samtools depth -a -b "$SCRATCH/sites.bed" "$BAM_DIR/A.bam" | sort -k3,3n | awk '{print $1":"$2, $3}' | sed -n '1p;$p'

chr5:177515944 1023
chr21:43104346 1376


The first run gave 1,023 at DDX41 (chr5:177,515,944) and 1,376 at U2AF1 (chr21:43,104,346): every site well under 2,000 and far above BAMSurgeon's minimum of 10. It takes a minute because -b reads through the whole BAM rather than jumping to each site.

In [34]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to spike Sample B again.
REDO=false

# Set output and done marker (to skip if exists)
# Check done as marker and output file exist (in scratch) and truth.vcf.gz (in WORK)
# So different places - confirms it was run, BAM hasn't been removed to make room,
# (BAMs aren't moved to WORK so not saved) so need to be rebuilt each session
OUT="$BAM_DIR/B.spiked.bam"
DONE="$BAM_DIR/B.done"
if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$OUT" ] && [ -s "$TRUTH_DIR/truth.vcf.gz" ]; then
  echo "B.spiked.bam already present: skipped."
  exit 0
fi
[ -s "$BAM_DIR/A.bam" ] || { echo "No A.bam: run step 5 first (BAMs in scratch don't survive a session)."; exit 1; }
[ -s "$TRUTH_DIR/spike.vcf" ] || { echo "No spike.vcf: run step 2 first."; exit 1; }
rm -f "$DONE"
SECONDS=0

# BAMSurgeon writes stray files to the folder it runs in; keep them in scratch.
cd "$SCRATCH"
# Clear BAMSurgeon's temporary and log folders from any earlier run this session.
rm -rf "$SCRATCH"/bamsurgeon.tmp* "$SCRATCH"/bamsurgeon_logs_*

# Spike the truth variants into Sample A's own reads.
#   -v spike.vcf          variants to add; each record's INFO/VAF sets the
#                         fraction of reads at that site to edit.
#   -f A.bam  -r REF      reads to edit; reference (faidx'd and bwa-indexed).
#   -o                    output BAM. Despite its header, it is NOT sorted.
#   --aligner mem         realign edited reads with bwa mem, as the pipeline does.
#                         The default (old bwa aln) suits short reads only.
#   --alignerthreads      threads for each realignment.
#   -p 1                  ONE site at a time. Each realignment is a separate bwa
#                         run that loads the whole ~5.5 GB index; two at once
#                         would need ~11 GB of Colab's ~12.7 GB.
#   --seed 42             reproducible read selection.
#   --tmpdir              scratch folder (default: bamsurgeon.tmp here).
#   --vcf TRUTH_DIR/      where the truth VCF goes; the trailing / makes it a
#                         folder. Name: <outbam>.bamsurgeon.<varfile>.vcf
# Defaults worth knowing: sites with under 10 or over 2,000 reads are skipped,
# and at least 3 reads must carry the variant.
# tee shows progress here and keeps a copy in the log.
echo "Spiking 21 variants, one at a time (about 20 minutes on free Colab)..."
bamsurgeon -v "$TRUTH_DIR/spike.vcf" -f "$BAM_DIR/A.bam" -r "$REF" -o "$SCRATCH/B.unsorted.bam" \
  --aligner mem --alignerthreads "$THREADS" -p 1 --seed 42 \
  --tmpdir "$SCRATCH/bamsurgeon.tmp" --vcf "$TRUTH_DIR/" 2>&1 | tee "$LOGS_DIR/step6_bamsurgeon.log"

# Sort and index (samtools index refuses an unsorted BAM), then drop the unsorted copy.
samtools sort -@ "$THREADS" -T "$SCRATCH/sortB" -o "$OUT" "$SCRATCH/B.unsorted.bam"
samtools index "$OUT"
rm -f "$SCRATCH/B.unsorted.bam"

# Check every variant landed at roughly the requested VAF. The script comes
# with BAMSurgeon's code (section 8) and writes one row per site:
#   --orig-bam  the BAM before spiking, for depth comparisons
#   --tsv       per-site report: OK, LOW_VAF, ABSENT or NO_COVERAGE
# It judges an indel by how its reads are aligned, which misjudges the two large
# ones (see "What the first run showed" under this cell). So its own pass/fail
# is ignored here (|| true), and the cell decides below instead.
TRUTH_RAW="$TRUTH_DIR/B.unsorted.bamsurgeon.spike.vcf"
python3 "$BAMSURGEON_DIR/scripts/validate_spikein.py" --vcf "$TRUTH_RAW" \
  --bam "$OUT" --orig-bam "$BAM_DIR/A.bam" --ref "$REF" --tsv "$QC_DIR/spikein_report.tsv" || true

# 1) Every site except the CALR deletion and the FLT3-ITD must read OK.
#    Report columns: chrom pos kind req_vaf depth alt obs_vaf status note.
NOT_OK=$(awk -F'\t' 'NR > 1 && $8 != "OK" && !($1 == "chr19" && $2 == 12943750) && !($1 == "chr13" && $2 == 28034100)' \
  "$QC_DIR/spikein_report.tsv")

# 2) The two large indels: count reads whose raw bases (soft-clipped ones
#    included) carry the variant's junction, a 30 bp sequence found only in
#    mutant reads, or the matching wild-type 30 bp. Pass if the VAF is within
#    0.6-1.4x of the requested one. -F 0x900 skips secondary and supplementary
#    records, so no read is counted twice.
seqof() { samtools faidx "$REF" "$1" | tail -n +2 | tr -d '\n' | tr 'acgt' 'ACGT'; }
junction() {  # junction NAME REGION MUTANT_30BP WILDTYPE_30BP REQUESTED_VAF
  local m w
  m=$(samtools view -F 0x900 "$OUT" "$2" | awk -v s="$3" 'index($10, s)' | wc -l)
  w=$(samtools view -F 0x900 "$OUT" "$2" | awk -v s="$4" 'index($10, s)' | wc -l)
  awk -v n="$1" -v m="$m" -v w="$w" -v r="$5" 'BEGIN {
    v = (m + w > 0) ? m / (m + w) : 0
    ok = (v >= 0.6 * r && v <= 1.4 * r)
    printf "%s\t%d\t%d\t%.3f\t%.2f\t%s\n", n, m, w, v, r, (ok ? "OK" : "OUT_OF_RANGE")
    exit (ok ? 0 : 1) }'
}
JFILE="$QC_DIR/spikein_junctions.tsv"
printf 'name\tcarrying\twild_type\tvaf\trequested\tstatus\n' > "$JFILE"
JFAIL=0
# CALR: 52 bp deleted after chr19:12,943,750. Mutant = the 15 bp before the
# gap joined to the 15 bp after it; wild type = the same 15 bp + the first 15
# deleted bases.
L=$(seqof chr19:12943736-12943750)
junction CALR_type1 chr19:12943500-12944050 "$L$(seqof chr19:12943803-12943817)" \
  "$L$(seqof chr19:12943751-12943765)" 0.10 >> "$JFILE" || JFAIL=1
# FLT3-ITD: a copy of chr13:28,034,101-28,034,130 inserted after 28,034,130.
# Mutant = the stretch's last 15 bp joined to its first 15; wild type = its
# last 15 bp + the next 15 bp of reference.
E=$(seqof chr13:28034116-28034130)
junction FLT3_ITD30 chr13:28033850-28034400 "$E$(seqof chr13:28034101-28034115)" \
  "$E$(seqof chr13:28034131-28034145)" 0.05 >> "$JFILE" || JFAIL=1

echo "Report status counts (column 8; the CALR and ITD rows are judged by the junction check):"
cut -f8 "$QC_DIR/spikein_report.tsv" | grep -v "^status$" | sort | uniq -c
echo "Junction check for the two large indels:"
cat "$JFILE"
if [ -n "$NOT_OK" ] || [ "$JFAIL" -ne 0 ]; then
  if [ -n "$NOT_OK" ]; then echo "Sites not OK:"; echo "$NOT_OK"; fi
  echo "Spike-in check failed: see above. B.done not written."
  exit 1
fi

# 3) Sort, compress and index the truth VCF; vcfeval (step 9) needs all three.
#    BAMSurgeon processes SNVs first and indels last, and tabix refuses records
#    out of position order, so sort first: header lines, then records by
#    chromosome and numeric position.
#    bgzip -c  block-compress to stdout   tabix -p vcf  index it as a VCF
#    tabix -f  overwrite an index from an earlier run
{ grep '^#' "$TRUTH_RAW"; grep -v '^#' "$TRUTH_RAW" | sort -k1,1 -k2,2n; } | bgzip -c > "$TRUTH_DIR/truth.vcf.gz"
tabix -f -p vcf "$TRUTH_DIR/truth.vcf.gz"
touch "$DONE"

echo "Done in $((SECONDS / 60)) min. Truth records: $(bcftools view -H "$TRUTH_DIR/truth.vcf.gz" | wc -l) (expect 21)"

Spiking 21 variants, one at a time (about 20 minutes on free Colab)...
INFO 2026-11-23 01:47:07,214 starting /content/micromamba/envs/bioinf_portfolio/bin/bamsurgeon called with args: /content/micromamba/envs/bioinf_portfolio/bin/bamsurgeon -v /content/my_project/data/raw/truth/spike.vcf -f /content/scratch/bam/A.bam -r /content/my_project/reference/genome/hg38_masked.fa -o /content/scratch/B.unsorted.bam --aligner mem --alignerthreads 2 -p 1 --seed 42 --tmpdir /content/scratch/bamsurgeon.tmp --vcf /content/my_project/data/raw/truth/
INFO 2026-11-23 01:47:07,220 read 0 SV and 21 small variant request(s)
INFO 2026-11-23 01:47:07,229 applying 21 small variant(s) in 21 cluster(s)
INFO 2026-11-23 01:47:07,229 starting /content/micromamba/envs/bioinf_portfolio/bin/bamsurgeon called with args: /content/micromamba/envs/bioinf_portfolio/bin/bamsurgeon -v /content/my_project/data/raw/truth/spike.vcf -f /content/scratch/bam/A.bam -r /content/my_project/reference/genome/hg38_masked.fa -o /content

[bam_sort_core] merging from 1 files and 2 in-memory blocks...


Explicitly check that the CALR, FLT3_ITD spikes map correctly. Previously, bwa behaviour meant junctions were not being counted correctly. See output above indicating LOW_VAF flag at Chr19.

In [35]:
%%bash
B="$BAM_DIR/B.spiked.bam"
seqof() { samtools faidx "$REF" "$1" | tail -n +2 | tr -d '\n' | tr 'acgt' 'ACGT'; }
count() {  # count NAME REGION MUTANT_30BP WILDTYPE_30BP
  local m w
  m=$(samtools view -F 0x900 "$B" "$2" | awk -v s="$3" 'index($10, s)' | wc -l)
  w=$(samtools view -F 0x900 "$B" "$2" | awk -v s="$4" 'index($10, s)' | wc -l)
  awk -v n="$1" -v m="$m" -v w="$w" 'BEGIN{printf "%-11s carrying %4d   wild type %5d   VAF %.3f\n", n, m, w, m/(m+w)}'
}
# CALR: 15 bp before the deletion joined to 15 bp after it, versus the same 15 bp + the first 15 deleted.
L=$(seqof chr19:12943736-12943750)
count CALR_type1 chr19:12943500-12944050 "$L$(seqof chr19:12943803-12943817)" "$L$(seqof chr19:12943751-12943765)"
# FLT3-ITD: the copied stretch's last 15 bp joined to its first 15, versus its last 15 + the next 15 of reference.
E=$(seqof chr13:28034116-28034130)
count FLT3_ITD30 chr13:28033850-28034400 "$E$(seqof chr13:28034101-28034115)" "$E$(seqof chr13:28034131-28034145)"

CALR_type1  carrying   96   wild type   856   VAF 0.101
FLT3_ITD30  carrying   49   wild type   946   VAF 0.049


## STEP 7:

In [36]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to write Sample B's FASTQs again.
REDO=false

OUT1="$SIM_DIR/B_1.fq.gz"
OUT2="$SIM_DIR/B_2.fq.gz"
DONE="$SIM_DIR/B.done"
if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$OUT1" ] && [ -s "$OUT2" ]; then
  echo "Sample B FASTQs already present: skipped."
  exit 0
fi
[ -s "$BAM_DIR/B.spiked.bam" ] || { echo "No B.spiked.bam: run step 6 first (in this session)."; exit 1; }
rm -f "$DONE"

# samtools collate puts each read next to its mate, which samtools fastq needs
# to write matching R1 and R2 files.
#   -u  uncompressed output (faster when piping)   -O  write to stdout
#   -T  prefix for temporary files (scratch)       -@  extra threads
samtools collate -@ "$THREADS" -u -O -T "$SCRATCH/collateB" "$BAM_DIR/B.spiked.bam" |
  # Write the pairs out as FASTQ.
  #   -n           keep read names as they are (no /1 and /2 added)
  #   -1, -2       R1 and R2 outputs; .gz names make samtools compress them
  #   -0, -s       discard reads that are neither R1 nor R2, and singletons
  # Secondary and supplementary alignments are skipped by default, so each
  # read is written once.
  samtools fastq -@ "$THREADS" -n -1 "$OUT1" -2 "$OUT2" -0 /dev/null -s /dev/null
touch "$DONE"

# Sample B should have as many read pairs as Sample A.
echo "Read pairs: A $(zcat "$SIM_DIR/A_1.fq.gz" | awk 'NR % 4 == 1' | wc -l), B $(zcat "$OUT1" | awk 'NR % 4 == 1' | wc -l)"

Read pairs: A 3490000, B 3490000


[M::bam2fq_mainloop] discarded 0 singletons
[M::bam2fq_mainloop] processed 6980000 reads


# Two Sample Workflow: Test Pipeline

## Run Test Pipeline on A + B

This is the simplest pipeline that exercises the dataset: align, mark duplicates, call in tumour-only mode, filter, keep PASS calls, normalise. Swap in your own pipeline later and rerun step 9. It needs only the FASTQs, so it can start a fresh session.
*   In: both samples' FASTQs, the reference, the targets.
*   Out: `data/processed/calls/A.pass.vcf.gz` and `B.pass.vcf.gz`, both indexed; dedup BAMs in `BAM_DIR`.

In [38]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to rerun the pipeline on both samples.
REDO=false

# Reuse step 5's alignment for Sample A while it is still here. Step 5 ran the
# exact bwa command below (same reads, reference, read group and -K), so the
# result is identical, and this saves about 20 minutes (step 5 took 17).
# Even with re-use of the alignment for Sample A, this still takes ~45 minutes.
REUSE_STEP5_FOR_A=true

# Check if A.done or B.done exist, also checks A.pass.vcf.gz, and B.pass.vcg.gz
# Inner for loop checks simulated A and B _1,_2.fq.gz files exist.
# So if either at .done or .fq.gz files stages it fails, exits.
# From there, proceeds with 1) Align and sort, 2) Mark Dups, 3) Calls Variants,
# 4) Mutect 2 filtering, PASS filtering
for S in A B; do   # ${S} is A on the first pass, B on the second
  DONE="$CALLS_DIR/${S}.done"
  if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$CALLS_DIR/${S}.pass.vcf.gz" ]; then
    echo "${S}: calls already present, skipped."
    continue   # go on to the next sample
  fi
  for f in "$SIM_DIR/${S}_1.fq.gz" "$SIM_DIR/${S}_2.fq.gz"; do
    [ -s "$f" ] || { echo "Missing $f: run step 4 (A) or step 7 (B) first."; exit 1; }
  done
  rm -f "$DONE"
  SECONDS=0

  # 1) Align and sort, exactly as in step 5, with this sample's read group.
  #    Double quotes this time, so the shell fills in ${S}.
  if [ "$S" = A ] && [ "$REUSE_STEP5_FOR_A" = true ] && [ -f "$BAM_DIR/A.done" ]; then
    echo "A: reusing step 5's alignment."
    ALN="$BAM_DIR/A.bam"
  else
    echo "${S}: aligning (about 20 minutes; step 5 took 17)..."
    ALN="$BAM_DIR/${S}.aln.bam"
    bwa mem -t "$THREADS" -K 100000000 -R "@RG\tID:${S}\tSM:${S}\tPL:ILLUMINA" \
      "$REF" "$SIM_DIR/${S}_1.fq.gz" "$SIM_DIR/${S}_2.fq.gz" 2> "$LOGS_DIR/step8_bwa_${S}.log" |
      samtools sort -@ "$THREADS" -T "$SCRATCH/sort${S}" -o "$ALN"
  fi

  # 2) Mark PCR duplicates: pairs that start and end at the same positions
  #    probably copy one original molecule, and counting them twice would
  #    inflate the evidence. ART makes no PCR copies, but at 1000x a few pairs
  #    line up by chance; real data needs this step much more.
  #      --java-options  Java memory (JAVA_MEM from the setup cell)
  #      -M              duplicate metrics file   --CREATE_INDEX  write the .bai
  #      --TMP_DIR       temporary files in scratch
  gatk --java-options "$JAVA_MEM" MarkDuplicates -I "$ALN" -O "$BAM_DIR/${S}.dedup.bam" \
    -M "$QC_DIR/${S}.dup_metrics.txt" --CREATE_INDEX true --TMP_DIR "$SCRATCH"

  # 3) Call variants in tumour-only mode: no matched normal is given.
  #      -L      call only inside the exon targets
  #      -ip 50  pad each target by 50 bp, so exon-edge and splice-site
  #              variants get read context on both sides
  #      --native-pair-hmm-threads  threads for Mutect2's slowest calculation
  #    Left out to keep the minimum small: --germline-resource (gnomAD) and
  #    --panel-of-normals. Without them Mutect2 can't flag likely germline
  #    variants, so the 50% germline-like rows come through as PASS.
  gatk --java-options "$JAVA_MEM" Mutect2 -R "$REF" -I "$BAM_DIR/${S}.dedup.bam" \
    -L "$TARGETS_BED" -ip 50 --native-pair-hmm-threads "$THREADS" --tmp-dir "$SCRATCH" \
    -O "$CALLS_DIR/${S}.unfiltered.vcf.gz"

  # 4) Mutect2's statistical filters, using the .stats file it wrote beside
  #    its VCF. Calls get PASS or the name of the filter they failed.
  gatk --java-options "$JAVA_MEM" FilterMutectCalls -R "$REF" -V "$CALLS_DIR/${S}.unfiltered.vcf.gz" \
    --tmp-dir "$SCRATCH" -O "$CALLS_DIR/${S}.filtered.vcf.gz"

  # 5) Keep PASS calls (-f PASS) and normalise them the same way as the truth
  #    set, so both files describe each variant the same way. -Oz = bgzipped VCF.
  #    Note: Tabix needs files sorted by position.
  bcftools view -f PASS "$CALLS_DIR/${S}.filtered.vcf.gz" | # pipe lines with filter column = PASS to next line
    bcftools norm -f "$REF" -m -both -Oz -o "$CALLS_DIR/${S}.pass.vcf.gz" # norm normalises to standard form, using -f "$REF" as reference
  tabix -f -p vcf "$CALLS_DIR/${S}.pass.vcf.gz"

  touch "$DONE"
  echo "${S}: done in $((SECONDS / 60)) min, $(bcftools view -H "$CALLS_DIR/${S}.pass.vcf.gz" | wc -l) PASS calls."
done

A: reusing step 5's alignment.
Tool returned:
0
Tool returned:
SUCCESS
A: done in 8 min, 0 PASS calls.
B: aligning (about 20 minutes; step 5 took 17)...
Tool returned:
0
Tool returned:
SUCCESS
B: done in 36 min, 16 PASS calls.


Using GATK jar /content/micromamba/envs/bioinf_portfolio/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4g -jar /content/micromamba/envs/bioinf_portfolio/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar MarkDuplicates -I /content/scratch/bam/A.bam -O /content/scratch/bam/A.dedup.bam -M /content/my_project/output/results/qc/A.dup_metrics.txt --CREATE_INDEX true --TMP_DIR /content/scratch
02:09:44.764 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/micromamba/envs/bioinf_portfolio/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Mon Nov 23 02:09:44 UTC 2026] MarkDuplicates --INPUT /content/scratch/bam/A.bam --OUTPUT /content/scratch/bam/A.dedup.bam --METRICS_FILE /content/my_project/output/results/qc/A.dup_metrics.txt --TMP

## Score and make results table

`vcfeval` matches calls to truth by haplotype, so an indel written at a different but equivalent position still counts. `--squash-ploidy` ignores genotype, which suits somatic calls. The cell then writes a small base-R script beside the results and runs it, so your methods can point to the exact code.
*  In: `truth.vcf.gz`, both PASS VCFs, the RTG SDF, the targets, variants.norm.tsv.
*  Out: `output/results/eval/B/` (summary plus true-positive, false-negative and false-positive VCFs), false-positive counts for A and B, `B_results_by_variant.tsv` and `results_table.R`.
*  Then save: Run Save cell

In [40]:
%%bash
set -euo pipefail

# Check files and SDF
for f in "$TRUTH_DIR/truth.vcf.gz" "$CALLS_DIR/A.pass.vcf.gz" "$CALLS_DIR/B.pass.vcf.gz" "$TRUTH_DIR/variants.norm.tsv"; do
  [ -s "$f" ] || { echo "Missing $f: run steps 2, 6 and 8 first."; exit 1; }
done
[ -d "$SDF" ] || { echo "No RTG SDF: run step 0 first."; exit 1; }

# vcfeval refuses an output folder that already exists, so clear the last
# scoring first. (-u at the top of script stops the cell if EVAL_DIR were
# unset, so this can never become rm -rf /B.)
rm -rf "$EVAL_DIR/B"

# Score Sample B's PASS calls against the truth.
#   -b truth (what should be found)   -c calls (what the pipeline found)
#   -t SDF  reference in RTG's format   -e  score only inside the targets
#   --squash-ploidy  match on the allele and ignore genotype (0/1 vs 1/1),
#                    which carries little meaning for somatic calls
#   -o  output folder
# vcfeval matches by haplotype, so an indel written at a different but
# equivalent position still counts.
rtg vcfeval -b "$TRUTH_DIR/truth.vcf.gz" -c "$CALLS_DIR/B.pass.vcf.gz" -t "$SDF" \
  -e "$TARGETS_BED" --squash-ploidy -o "$EVAL_DIR/B"
cat "$EVAL_DIR/B/summary.txt"

# False positives in the negative sample: any PASS call inside the targets.
#   -H  records without the header   -R  only records inside these regions
echo "False positives, Sample A: $(bcftools view -H -R "$TARGETS_BED" "$CALLS_DIR/A.pass.vcf.gz" | wc -l)"
# False positives in Sample B: calls vcfeval couldn't match to any truth variant.
FP_B=$(bcftools view -H "$EVAL_DIR/B/fp.vcf.gz" | wc -l)
echo "False positives, Sample B: $FP_B"

# Where Sample B's false positives come from. B is Sample A's reads with 42,278
# of them edited (step 6), so a false call that A also has is background:
# simulated sequencing errors or alignment. One only in B sits near a spike
# site and was probably caused by the editing or realignment.
#   bcftools isec -n=2 -w1  records in both files, written as file 1 has them
#   bcftools isec -C -w1    records only in file 1
#   -c none                 a match needs the same position, REF and ALT
# awk '!/^#/' drops header lines; unlike grep, it never fails on zero matches.
FP_SHARED=$(bcftools isec -n=2 -w1 -c none "$EVAL_DIR/B/fp.vcf.gz" "$CALLS_DIR/A.pass.vcf.gz" | awk '!/^#/' | wc -l)
echo "  also called in Sample A (shared background): $FP_SHARED"
echo "  only in Sample B (probably from spiking):    $((FP_B - FP_SHARED))"
bcftools isec -C -w1 -c none "$EVAL_DIR/B/fp.vcf.gz" "$CALLS_DIR/A.pass.vcf.gz" |
  awk '!/^#/ {print "    " $1 ":" $2 " " $4 ">" $5}'

# One row per truth variant, found or missed, with the VAF BAMSurgeon achieved.
#   tp-baseline.vcf.gz  truth variants found   fn.vcf.gz  truth variants missed
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\tfound\n'  "$EVAL_DIR/B/tp-baseline.vcf.gz" >  "$EVAL_DIR/B_by_variant.tsv"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\tmissed\n' "$EVAL_DIR/B/fn.vcf.gz"          >> "$EVAL_DIR/B_by_variant.tsv"

# The results table, in base R. The script is saved beside the results, so
# your methods can point to it. Quoted 'EOF': the text is written as is.
cat > "$EVAL_DIR/results_table.R" <<'EOF'
# Folders come from the notebook's environment variables.
eval_dir  <- Sys.getenv("EVAL_DIR")
truth_dir <- Sys.getenv("TRUTH_DIR")

# Both files are tab-separated with no header row, so name the columns here.
res <- read.delim(file.path(eval_dir, "B_by_variant.tsv"), header = FALSE,
                  col.names = c("CHROM", "POS", "REF", "ALT", "VAF_achieved", "status"))
truth <- read.delim(file.path(truth_dir, "variants.norm.tsv"), header = FALSE,
                    col.names = c("CHROM", "POS", "REF", "ALT", "VAF_target", "NAME"))

# SNV if REF and ALT are both one base; otherwise an indel (including the ITD).
truth$class <- ifelse(nchar(truth$REF) == 1 & nchar(truth$ALT) == 1, "SNV", "indel")

# Join on position. Both files carry normalised positions, so they line up.
# all.x = TRUE keeps every truth variant even if it has no match.
m <- merge(truth, res[, c("CHROM", "POS", "VAF_achieved", "status")],
           by = c("CHROM", "POS"), all.x = TRUE)
# %in% treats a missing status (NA) as FALSE, so an unmatched row counts as
# not found rather than breaking the sums.
m$found <- m$status %in% "found"

cat("\nPer variant:\n")
print(m[order(-m$VAF_target, m$NAME), c("NAME", "class", "VAF_target", "VAF_achieved", "status")],
      row.names = FALSE)
if (any(is.na(m$status))) cat("\nNA status = no match on position: check that row by hand.\n")

# Found / total for each tier and class, written as "3 of 3".
cat("\nBy tier:\n")
print(aggregate(found ~ VAF_target + class, data = m,
                FUN = function(x) paste(sum(x), "of", length(x))), row.names = FALSE)

write.table(m, file.path(eval_dir, "B_results_by_variant.tsv"), sep = "\t",
            quote = FALSE, row.names = FALSE)
EOF
Rscript "$EVAL_DIR/results_table.R"

Threshold  True-pos-baseline  True-pos-call  False-pos  False-neg  Precision  Sensitivity  F-measure
----------------------------------------------------------------------------------------------------
     None                 16             16          0          5     1.0000       0.7619     0.8649

Threshold  True-pos-baseline  True-pos-call  False-pos  False-neg  Precision  Sensitivity  F-measure
----------------------------------------------------------------------------------------------------
     None                 16             16          0          5     1.0000       0.7619     0.8649
False positives, Sample A: 0
False positives, Sample B: 0
  also called in Sample A (shared background): 0
  only in Sample B (probably from spiking):    0

Per variant:
            NAME class VAF_target VAF_achieved status
  DDX41_D140Gfs2 indel       0.50    0.4897360 missed
      FLT3_T227M   SNV       0.50    0.4995760 missed
     TET2_I1762V   SNV       0.50    0.5004580 missed
       

Reference sequence chr4 is used in baseline but not in calls.
Reference sequence chr13 is used in baseline but not in calls.
There were 16 variants not thresholded in ROC data files due to missing or invalid GQ (FORMAT) values.
Could not select maximized F-measure threshold from ROC data, only un-thresholded statistics will be shown. Consider selecting a different scoring attribute with --vcf-score-field


Solving FLT3-ITD failure:

In [41]:
%%bash
echo "1) Mutect2 candidates near the ITD (REF and ALT shown as lengths):"
bcftools query -r chr13:28033950-28034300 -f '%POS\t%REF\t%ALT\t%FILTER\t[%AF]\n' "$CALLS_DIR/B.filtered.vcf.gz" |
  awk -F'\t' '{printf "  %s  %dbp>%dbp  %s  AF=%s\n", $1, length($2), length($3), $4, $5}'
echo "   (no lines = Mutect2 never proposed a variant here)"

echo "2) How bwa aligned Sample B's reads across the ITD (primary, non-duplicate):"
samtools view -F 0xF04 "$BAM_DIR/B.dedup.bam" chr13:28034095-28034135 |
  awk '{n++} $6 ~ /[0-9]+I/ {ins++} $6 ~ /S/ {clip++}
       END {printf "  %d reads: %d with an insertion in the CIGAR, %d soft-clipped\n", n, ins, clip}'

echo "3) Junction count in the pipeline's own alignment:"
seqof() { samtools faidx "$REF" "$1" | tail -n +2 | tr -d '\n' | tr 'acgt' 'ACGT'; }
E=$(seqof chr13:28034116-28034130)
M="$E$(seqof chr13:28034101-28034115)"; W="$E$(seqof chr13:28034131-28034145)"
m=$(samtools view -F 0xF04 "$BAM_DIR/B.dedup.bam" chr13:28033850-28034400 | awk -v s="$M" 'index($10, s)' | wc -l)
w=$(samtools view -F 0xF04 "$BAM_DIR/B.dedup.bam" chr13:28033850-28034400 | awk -v s="$W" 'index($10, s)' | wc -l)
echo "  carrying $m, wild type $w"

1) Mutect2 candidates near the ITD (REF and ALT shown as lengths):
  28034100  1bp>31bp  slippage  AF=0.049
   (no lines = Mutect2 never proposed a variant here)
2) How bwa aligned Sample B's reads across the ITD (primary, non-duplicate):
  1691 reads: 13 with an insertion in the CIGAR, 43 soft-clipped
3) Junction count in the pipeline's own alignment:
  carrying 48, wild type 935


In [42]:
%%bash
echo "Filter outcomes for Sample B's candidates:"
bcftools query -f '%FILTER\n' "$CALLS_DIR/B.filtered.vcf.gz" | sort | uniq -c | sort -rn
echo
echo "Truth sites in Mutect2's output (position, FILTER, allele fraction):"
bcftools query -T "$TRUTH_DIR/truth.vcf.gz" -f '%CHROM\t%POS\t%FILTER\t[%AF]\n' "$CALLS_DIR/B.filtered.vcf.gz"

Filter outcomes for Sample B's candidates:
     16 PASS
      4 germline
      1 slippage

Truth sites in Mutect2's output (position, FILTER, allele fraction):
chr1	114716126	PASS	0.02
chr2	25234373	PASS	0.017
chr2	197402110	PASS	0.371
chr2	208248388	PASS	0.092
chr3	38141150	PASS	0.096
chr3	49375540	PASS	0.02
chr4	105275794	germline	0.473
chr5	171410539	PASS	0.019
chr5	177515944	germline	0.454
chr7	140753336	PASS	0.046
chr9	5073770	PASS	0.364
chr12	25245350	PASS	0.02
chr13	28034100	slippage	0.049
chr13	28050157	germline	0.472
chr15	90088702	PASS	0.042
chr17	7673802	PASS	0.376
chr17	7676154	germline	0.47
chr17	76736877	PASS	0.09
chr19	12943750	PASS	0.065
chr20	32434638	PASS	0.339
chr21	43104346	PASS	0.047


In [43]:
%%bash
gatk --java-options "$JAVA_MEM" FilterMutectCalls -R "$REF" -V "$CALLS_DIR/B.unfiltered.vcf.gz" \
  --pcr-slippage-rate 0.01 --tmp-dir "$SCRATCH" -O "$SCRATCH/B.filtered.slip001.vcf.gz" 2> /dev/null
bcftools query -r chr13:28034100 -f '%POS\t%FILTER\t[%AF]\n' "$SCRATCH/B.filtered.slip001.vcf.gz"

28034100	PASS	0.049


# Three Sample Workflow: Build Simulation Files

Setup

In [46]:
import os
import shutil
from pathlib import Path

# Run in a fresh session: notebook 01 sections 1-9 first (SETTINGS, PATH,
# BAMSurgeon), then this notebook. Don't mix in notebook 02's cells: both
# notebooks use the same variable names for different folders.
if "SETTINGS" not in globals():
    raise RuntimeError("SETTINGS is missing: run notebook 01 sections 1-9 first.")
for key in ("WORK", "LOGS_DIR", "BAMSURGEON_DIR"):
    if not os.environ.get(key):
        raise RuntimeError(f"{key} is not set: run notebook 01 sections 1-9 first.")

RUN = "run2"   # this run's folder name; a later run gets a new name
WORK = SETTINGS["WORK"]

# The sample sheet. Every step from 4 on loops over it.
#   negative  reads simulated by ART with art_seed
#   positive  a copy of base's reads with truth set `truth` spiked in by
#             BAMSurgeon, choosing reads with spike_seed
# A and B repeat run 1 with its seeds, so step 10 can check run 1 is
# reproduced. Delete those two rows to skip that rerun (about 2 hours).
SAMPLES = [
    # sample  kind        art_seed  base   truth  spike_seed
    ("A",     "negative", "42",     ".",   ".",   "."),
    ("B",     "positive", ".",      "A",   "B",   "42"),
    ("A2",    "negative", "43",     ".",   ".",   "."),
    ("C",     "positive", ".",      "A2",  "C",   "43"),
]

# Check the sheet before anything uses it.
names = [row[0] for row in SAMPLES]
if len(set(names)) != len(names):
    raise RuntimeError(f"Sample names must be unique: {names}")
negatives = set()
for name, kind, art_seed, base, truth, spike_seed in SAMPLES:
    if kind == "negative" and art_seed.isdigit():
        negatives.add(name)
    elif kind == "positive" and base in negatives and truth != "." and spike_seed.isdigit():
        pass
    else:
        raise RuntimeError(f"Sample {name}: a negative needs a numeric art_seed; a positive "
                           f"needs a base listed above it as a negative, a truth set and a spike_seed.")

RUN_DIR = f"{WORK}/runs/{RUN}"
NB3 = {
    # Shared inputs, built once by notebook 02 (steps 0, 1 and 3). Read only.
    "REF_DIR": f"{WORK}/reference/genome",
    "DESIGN_DIR": f"{WORK}/reference/sureselect",
    "PANEL_SRC": f"{WORK}/data/raw/sim/panel_src.fa",     # ART's input (step 3)
    # This run. Everything it writes goes under RUN_DIR (saved to Drive by
    # notebook 01 section 11) or under SCRATCH (never saved).
    "RUN": RUN,
    "RUN_DIR": RUN_DIR,
    "SAMPLE_SHEET": f"{RUN_DIR}/samples.tsv",
    "SHEET_SH": f"{RUN_DIR}/sheet.sh",                    # helpers for the bash cells
    "TRUTH_DIR": f"{RUN_DIR}/truth",                      # one folder per truth set
    "SIM_DIR": f"{RUN_DIR}/sim",                          # FASTQs (steps 4, 7)
    "CALLS_DIR": f"{RUN_DIR}/calls",                      # pipeline VCFs (step 8)
    "QC_DIR": f"{RUN_DIR}/qc",                            # depth, capture, spike-in checks
    "EVAL_DIR": f"{RUN_DIR}/eval",                        # scoring (step 9)
    "RUN_LOGS": f"{RUN_DIR}/logs",                        # this run's tool logs
    "SCRATCH": "/content/scratch",
    "BAM_DIR": f"/content/scratch/{RUN}/bam",             # BAMs: local only, rebuilt if lost
    # Run 1's outputs (notebook 02's folders), for step 10's comparison.
    "RUN1_SIM": f"{WORK}/data/raw/sim",
    "RUN1_TRUTH": f"{WORK}/data/raw/truth",
    "RUN1_CALLS": f"{WORK}/data/processed/calls",
    "RUN1_EVAL": f"{WORK}/output/results/eval",
}
NB3["REF"] = f"{NB3['REF_DIR']}/hg38_masked.fa"
NB3["SDF"] = f"{NB3['REF_DIR']}/hg38_masked.sdf"
NB3["GENOME"] = f"{NB3['REF_DIR']}/hg38.genome"
NB3["TARGETS_BED"] = f"{NB3['DESIGN_DIR']}/Regions_SNV.clean.bed"
NB3["BAITS_BED"] = f"{NB3['DESIGN_DIR']}/Covered_SNV.clean.bed"
NB3["TARGETS_IL"] = f"{NB3['DESIGN_DIR']}/targets.interval_list"
NB3["BAITS_IL"] = f"{NB3['DESIGN_DIR']}/baits.interval_list"
NB3["THREADS"] = str(os.cpu_count() or 2)
NB3["JAVA_MEM"] = "-Xmx4g"

# Notebook 02's shared outputs must already exist.
needed = [NB3["REF"], NB3["REF"] + ".fai", NB3["REF"] + ".bwt", str(Path(NB3["REF"]).with_suffix(".dict")),
          NB3["GENOME"], NB3["TARGETS_BED"], NB3["BAITS_BED"], NB3["TARGETS_IL"], NB3["BAITS_IL"],
          NB3["PANEL_SRC"]]
missing = [p for p in needed if not (os.path.isfile(p) and os.path.getsize(p) > 0)]
if not os.path.isdir(NB3["SDF"]):
    missing.append(NB3["SDF"])
if missing:
    raise RuntimeError("Missing (run notebook 02 steps 0, 1 and 3 once, and save them):\n  " + "\n  ".join(missing))

os.environ.update(NB3)
for key in ("TRUTH_DIR", "SIM_DIR", "CALLS_DIR", "QC_DIR", "EVAL_DIR", "RUN_LOGS", "BAM_DIR"):
    Path(NB3[key]).mkdir(parents=True, exist_ok=True)

# Write the sheet as a tab-separated file, for the bash and R cells.
with open(NB3["SAMPLE_SHEET"], "w") as out:
    out.write("sample\tkind\tart_seed\tbase\ttruth\tspike_seed\n")
    for row in SAMPLES:
        out.write("\t".join(row) + "\n")

# Helpers every bash cell loads with `source "$SHEET_SH"`:
#   samples KIND        the samples of one kind, in sheet order
#   get SAMPLE COLUMN   one value from a sample's row, by column name
#   truth_sets          the truth sets the positives use, once each
with open(NB3["SHEET_SH"], "w") as out:
    out.write(r'''# Sample-sheet helpers (written by notebook 03's setup cell).
samples() { awk -F'\t' -v k="$1" 'NR > 1 && $2 == k {print $1}' "$SAMPLE_SHEET"; }
truth_sets() { awk -F'\t' 'NR > 1 && $2 == "positive" && !seen[$5]++ {print $5}' "$SAMPLE_SHEET"; }
get() {
  awk -F'\t' -v s="$1" -v c="$2" '
    NR == 1 { for (i = 1; i <= NF; i++) col[$i] = i
              if (!(c in col)) { print "sheet.sh: no column " c > "/dev/stderr"; bad = 1; exit 1 }
              next }
    $1 == s { print $col[c]; found = 1 }
    END     { if (bad) exit 1
              if (!found) { print "sheet.sh: no sample " s > "/dev/stderr"; exit 1 } }' "$SAMPLE_SHEET"
}
''')

tools = ["samtools", "bcftools", "bgzip", "tabix", "bwa", "art_illumina", "mosdepth",
         "picard", "gatk", "rtg", "bamsurgeon", "Rscript", "md5sum"]
missing = [t for t in tools if shutil.which(t) is None]
if missing:
    raise RuntimeError(f"Not found on PATH: {missing}. Run notebook 01 sections 6-8 first.")

print(f"Run {RUN}: {RUN_DIR}")
for name, kind, art_seed, base, truth, spike_seed in SAMPLES:
    made = f"ART seed {art_seed}" if kind == "negative" else f"{base} + truth set {truth}, spike seed {spike_seed}"
    print(f"  {name:4} {kind:9} {made}")
print("Shared inputs found. All tools found.")

Run run2: /content/my_project/runs/run2
  A    negative  ART seed 42
  B    positive  A + truth set B, spike seed 42
  A2   negative  ART seed 43
  C    positive  A2 + truth set C, spike seed 43
Shared inputs found. All tools found.


In [47]:
import os
import subprocess
from pathlib import Path

REF = os.environ["REF"]
TRUTH_DIR = Path(os.environ["TRUTH_DIR"])

# Truth set B: run 1's table, unchanged (notebook 02 step 2 has its notes).
CALR_DEL52 = "GCAGAGGCTTAAGGAGGAGGAAGAAGACAAGAAACGCAAAGAGGAGGAGGAG"
TRUTH_B = [
    # name            VAF   chrom    position   REF                ALT      MANE Select       HGVS c.           HGVS p.               ID
    ("JAK2_V617F",     0.40, "chr9",  5073770,   "G",               "T",     "NM_004972.4",    "c.1849G>T",      "p.Val617Phe",        "rs77375493"),
    ("SF3B1_K700E",    0.40, "chr2",  197402110, "T",               "C",     "NM_012433.4",    "c.2098A>G",      "p.Lys700Glu",        "rs559063155"),
    ("TP53_R273H",     0.40, "chr17", 7673802,   "C",               "T",     "NM_000546.6",    "c.818G>A",       "p.Arg273His",        "rs28934576"),
    ("ASXL1_G646Wfs12", 0.40, "chr20", 32434638, "A",               "AG",    "NM_015338.6",    "c.1934dup",      "p.Gly646TrpfsTer12", "ClinVar:426927"),
    ("IDH1_R132H",     0.10, "chr2",  208248388, "C",               "T",     "NM_005896.4",    "c.395G>A",       "p.Arg132His",        "rs121913500"),
    ("SRSF2_P95H",     0.10, "chr17", 76736877,  "G",               "T",     "NM_001195427.2", "c.284C>A",       "p.Pro95His",         "rs751713049"),
    ("MYD88_L265P",    0.10, "chr3",  38141150,  "T",               "C",     "NM_002468.5",    "c.755T>C",       "p.Leu252Pro",        "rs387907272"),
    ("CALR_type1",     0.10, "chr19", 12943750,  "A" + CALR_DEL52,  "A",     "NM_004343.4",    "c.1099_1150del", "p.Leu367ThrfsTer46", "rs1555760738"),
    ("IDH2_R140Q",     0.05, "chr15", 90088702,  "C",               "T",     "NM_002168.4",    "c.419G>A",       "p.Arg140Gln",        "rs121913502"),
    ("BRAF_V600E",     0.05, "chr7",  140753336, "A",               "T",     "NM_004333.6",    "c.1799T>A",      "p.Val600Glu",        "rs113488022"),
    ("U2AF1_S34F",     0.05, "chr21", 43104346,  "G",               "A",     "NM_006758.3",    "c.101C>T",       "p.Ser34Phe",         "rs371769427"),
    ("NRAS_G12D",      0.02, "chr1",  114716126, "C",               "T",     "NM_002524.5",    "c.35G>A",        "p.Gly12Asp",         "rs121913237"),
    ("KRAS_G12D",      0.02, "chr12", 25245350,  "C",               "T",     "NM_004985.5",    "c.35G>A",        "p.Gly12Asp",         "rs121913529"),
    ("DNMT3A_R882H",   0.02, "chr2",  25234373,  "C",               "T",     "NM_022552.5",    "c.2645G>A",      "p.Arg882His",        "rs147001633"),
    ("RHOA_G17V",      0.02, "chr3",  49375540,  "C",               "A",     "NM_001664.4",    "c.50G>T",        "p.Gly17Val",         "rs11552761"),
    ("NPM1_typeA",     0.02, "chr5",  171410539, "C",               "CTCTG", "NM_002520.7",    "c.860_863dup",   "p.Trp288CysfsTer12", "rs587776806"),
    ("DDX41_D140Gfs2", 0.50, "chr5",  177515944, "T",               "TCATC", "NM_016222.4",    "c.415_418dup",   "p.Asp140GlyfsTer2",  "rs762890562"),
    ("TP53_P72R",      0.50, "chr17", 7676154,   "G",               "C",     "NM_000546.6",    "c.215C>G",       "p.Pro72Arg",         "rs1042522"),
    ("TET2_I1762V",    0.50, "chr4",  105275794, "A",               "G",     "NM_001127208.3", "c.5284A>G",      "p.Ile1762Val",       "rs2454206"),
    ("FLT3_T227M",     0.50, "chr13", 28050157,  "G",               "A",     "NM_004119.3",    "c.680C>T",       "p.Thr227Met",        "rs1933437"),
]

# Truth set C: new variants, checked on GeneBe on 23 Nov 2026 (MANE Select HGVS).
# Only the FLT3-ITD is shared with B, at 20% instead of 5% (see "Truth set C").
TRUTH_C = [
    ("CALR_type2",     0.10, "chr19", 12943813,  "A",               "ATTGTC", "NM_004343.4",   "c.1154_1155insTTGTC", "p.Lys385AsnfsTer47", "rs765476509"),
    ("JAK2_N542_E543del", 0.10, "chr9", 5070033, "GAAATGA",         "G",      "NM_004972.4",   "c.1624_1629del", "p.Asn542_Glu543del", "."),
    ("CEBPA_H24Afs",   0.10, "chr19", 33302346,  "C",               "CG",     "NM_004364.5",   "c.68dup",        "p.His24AlafsTer84",  "rs137852728"),
    ("MPL_W515L",      0.05, "chr1",  43349338,  "G",               "T",      "NM_005373.3",   "c.1544G>T",      "p.Trp515Leu",        "rs121913615"),
    ("FLT3_D835Y",     0.05, "chr13", 28018505,  "C",               "A",      "NM_004119.3",   "c.2503G>T",      "p.Asp835Tyr",        "rs121913488"),
    ("KIT_D816V",      0.02, "chr4",  54733155,  "A",               "T",      "NM_000222.3",   "c.2447A>T",      "p.Asp816Val",        "rs121913507"),
    ("TP53_R248Q",     0.01, "chr17", 7674220,   "C",               "T",      "NM_000546.6",   "c.743G>A",       "p.Arg248Gln",        "rs11540652"),
    ("SF3B1_K666N",    0.01, "chr2",  197402635, "C",               "A",      "NM_012433.4",   "c.1998G>T",      "p.Lys666Asn",        "rs377023736"),
    ("PTPN11_E76K",    0.005, "chr12", 112450406, "G",              "A",      "NM_002834.5",   "c.226G>A",       "p.Glu76Lys",         "rs121918464"),
    ("NRAS_Q61R",      0.005, "chr1",  114713908, "T",              "C",      "NM_002524.5",   "c.182A>G",       "p.Gln61Arg",         "rs11554290"),
]

TRUTH_SETS = {"B": TRUTH_B, "C": TRUTH_C}
ITD_VAF = {"B": 0.05, "C": 0.20}   # the FLT3-ITD's VAF in each set

# What each variant is expected to do, written down before the run. Step 9
# scores the outcome against it. Rules, first match wins:
#   VAF >= 0.45                  "germline": with no germline resource, Mutect2
#                                judges germline status by allele fraction alone
#   slippage test applies and    "slippage": below the filter's assumed 10%
#     VAF < 0.10                 slippage rate (notebook 02, step 9)
#   VAF <= 0.005                 "limit": the detection-limit probe; no expectation
#   otherwise                    "PASS"
SLIPPAGE_RATE = 0.10   # FilterMutectCalls --pcr-slippage-rate default
SLIPPAGE_MIN = 8       # FilterMutectCalls --min-pcr-slippage-size default (bases)


def ref_seq(chrom, start, end):
    """Bases chrom:start-end (1-based, inclusive) from the masked reference."""
    out = subprocess.run(["samtools", "faidx", REF, f"{chrom}:{start}-{end}"],
                         check=True, capture_output=True, text=True).stdout
    return "".join(out.splitlines()[1:]).upper()


TARGETS = {}
for line in open(os.environ["TARGETS_BED"]):
    chrom, start, end = line.split("\t")[:3]
    TARGETS.setdefault(chrom, []).append((int(start), int(end)))


def in_targets(chrom, pos):
    return any(start < pos <= end for start, end in TARGETS.get(chrom, []))


def period(s):
    """The shortest unit that s is made of: 'GG' -> 'G', 'CATC' -> 'CATC'."""
    for n in range(1, len(s) + 1):
        if len(s) % n == 0 and s[:n] * (len(s) // n) == s:
            return s[:n]


def slippage_test_applies(chrom, pos, ref, alt):
    """Approximates Mutect2's polymerase-slippage test. It applies to an indel
    that adds or removes exactly one copy of a repeat unit, where the reference
    holds at least SLIPPAGE_MIN bases of that repeat and both alleles keep at
    least one copy. Returns (applies, reference repeat length in bases)."""
    if len(ref) == len(alt):
        return False, 0
    changed = alt[1:] if len(alt) > len(ref) else ref[1:]
    unit = period(changed)
    if unit != changed:                      # more than one copy added or removed
        return False, 0
    after = ref_seq(chrom, pos + 1, pos + 1 + 20 * len(unit) + 200)
    copies = 0                               # copies in the reference, after the anchor
    while after.startswith(unit * (copies + 1)):
        copies += 1
    alt_copies = copies + 1 if len(alt) > len(ref) else copies - 1
    applies = copies * len(unit) >= SLIPPAGE_MIN and copies >= 1 and alt_copies >= 1
    return applies, copies * len(unit)


def kmers(seq, k):
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}


def junction_kmers(chrom, pos, ref, alt, flank=300):
    """A k-mer only the mutant allele has, and one only the reference allele has,
    each as near the variant as possible. Step 6 counts reads carrying each.
    k is 30, or 10 more than the indel's length, so a tandem duplication still
    leaves a k-mer unique to each allele."""
    k = max(30, abs(len(alt) - len(ref)) + 10)
    left = ref_seq(chrom, pos - flank, pos - 1)
    right = ref_seq(chrom, pos + len(ref), pos + len(ref) + flank - 1)
    wt, mut = left + ref + right, left + alt + right

    def nearest(seq, other, centre):
        other = kmers(other, k)
        best = None
        for i in range(len(seq) - k + 1):
            s = seq[i:i + k]
            if s not in other and (best is None or abs(i + k / 2 - centre) < best[0]):
                best = (abs(i + k / 2 - centre), s)
        return best[1] if best else None

    return (nearest(mut, wt, len(left) + len(alt) / 2),
            nearest(wt, mut, len(left) + len(ref) / 2), k)


def revcomp(s):
    return s[::-1].translate(str.maketrans("ACGT", "TGCA"))


def itd_row(vaf):
    # The FLT3-ITD from notebook 02: a second copy of chr13:28,034,101-28,034,130
    # (30 bp in exon 14). Written in the left-aligned form bcftools norm gave it
    # in run 1: the copy goes in after 28,034,100, in front of the original.
    anchor = ref_seq("chr13", 28034100, 28034100)
    dup = ref_seq("chr13", 28034101, 28034130)
    return ("FLT3_ITD30", vaf, "chr13", 28034100, anchor, anchor + dup, "NM_004119.3",
            "30 bp ITD (dup of chr13:28034101-28034130)", ".", ".")


# Build only the truth sets the sample sheet uses.
used = sorted({row[4] for row in SAMPLES if row[1] == "positive"})
unknown = [t for t in used if t not in TRUTH_SETS]
if unknown:
    raise RuntimeError(f"The sample sheet uses truth set(s) {unknown}, not defined here.")

failed = False
for tset in used:
    table = TRUTH_SETS[tset] + [itd_row(ITD_VAF[tset])]
    rows, problems = [], []
    print(f"Truth set {tset}:")
    for name, vaf, chrom, pos, ref, alt, tx, hgvs_c, hgvs_p, vid in table:
        found = ref_seq(chrom, pos, pos + len(ref) - 1)
        if found != ref:
            problems.append(f"{name}: reference has {found[:12]!r} at {chrom}:{pos}, table says {ref[:12]!r}")
        if not in_targets(chrom, pos):
            problems.append(f"{name}: {chrom}:{pos} is not inside a target")
        # New in notebook 03: an insertion must match its HGVS. A "dup" copies the
        # bases straight after the anchor; any other insertion must not (else its
        # HGVS would be a dup). Truth set B had only duplications.
        if len(alt) > len(ref):
            inserted = alt[len(ref):]
            is_copy = inserted == ref_seq(chrom, pos + 1, pos + len(inserted))
            is_dup = "dup" in hgvs_c
            if alt[0] != ref[0] or is_copy != is_dup:
                problems.append(f"{name}: inserted {inserted!r} {'is' if is_copy else 'is not'} "
                                f"a copy of the next bases, but HGVS is {hgvs_c}")
        elif len(ref) > len(alt) and alt != ref[0]:
            problems.append(f"{name}: a deletion's ALT must be the anchor base alone")
        slip, repeat_bp = slippage_test_applies(chrom, pos, ref, alt)
        if vaf >= 0.45:
            expect = "germline"
        elif slip and vaf < SLIPPAGE_RATE:
            expect = "slippage"
        elif vaf <= 0.005:
            expect = "limit"
        else:
            expect = "PASS"
        mut_k, wt_k, k = junction_kmers(chrom, pos, ref, alt)
        if mut_k is None or wt_k is None:
            problems.append(f"{name}: no {k}-mer tells the two alleles apart")
        rows.append((chrom, pos, ref, alt, vaf, name, tx, hgvs_c, hgvs_p, vid,
                     expect, "yes" if slip else "no", repeat_bp, mut_k, wt_k))
        shown = f"{len(ref) - 1} bp deleted" if len(ref) > 12 else (
                f"+{len(alt) - 1} bp" if len(alt) > 12 else f"{ref}>{alt}")
        print(f"  {name:18} {vaf:<6} {chrom + ':' + str(pos):16} {shown:14} "
              f"expect {expect:9} slippage test: {'yes, ' + str(repeat_bp) + ' bp repeat' if slip else 'no'}")

    rows.sort(key=lambda r: (r[0], r[1]))
    for a, b in zip(rows, rows[1:]):
        if a[0] == b[0] and b[1] - a[1] < 300:
            problems.append(f"{a[5]} and {b[5]} are only {b[1] - a[1]} bp apart (need >= 300)")
    if problems:
        print(f"  {len(problems)} problem(s):\n    " + "\n    ".join(problems))
        failed = True
        continue

    out_dir = TRUTH_DIR / tset
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "variants.tsv", "w") as out:        # for step 2's bash cell
        for r in rows:
            out.write("\t".join(str(x) for x in r[:6]) + "\n")
    with open(out_dir / "truth_set.tsv", "w") as out:       # annotation, for step 9
        out.write("chrom\tpos\tref\talt\tvaf\tname\tmane_select\thgvs_c\thgvs_p\tid\texpect\tslippage_test\trepeat_bp\n")
        for r in rows:
            out.write("\t".join(str(x) for x in r[:13]) + "\n")
    with open(out_dir / "junctions.tsv", "w") as out:       # for step 6's check
        out.write("name\tregion\tmutant\twild_type\tmutant_rc\twild_type_rc\tvaf\n")
        for chrom, pos, ref, alt, vaf, name, *_, mut_k, wt_k in rows:
            region = f"{chrom}:{pos - 300}-{pos + len(ref) + 300}"
            out.write("\t".join([name, region, mut_k, wt_k, revcomp(mut_k), revcomp(wt_k), str(vaf)]) + "\n")
    print(f"  {len(rows)} variants passed. Wrote variants.tsv, truth_set.tsv, junctions.tsv to {out_dir}")

if failed:
    raise RuntimeError("Fix the problems above, then rerun this cell.")

Truth set B:
  JAK2_V617F         0.4    chr9:5073770     G>T            expect PASS      slippage test: no
  SF3B1_K700E        0.4    chr2:197402110   T>C            expect PASS      slippage test: no
  TP53_R273H         0.4    chr17:7673802    C>T            expect PASS      slippage test: no
  ASXL1_G646Wfs12    0.4    chr20:32434638   A>AG           expect PASS      slippage test: yes, 8 bp repeat
  IDH1_R132H         0.1    chr2:208248388   C>T            expect PASS      slippage test: no
  SRSF2_P95H         0.1    chr17:76736877   G>T            expect PASS      slippage test: no
  MYD88_L265P        0.1    chr3:38141150    T>C            expect PASS      slippage test: no
  CALR_type1         0.1    chr19:12943750   52 bp deleted  expect PASS      slippage test: no
  IDH2_R140Q         0.05   chr15:90088702   C>T            expect PASS      slippage test: no
  BRAF_V600E         0.05   chr7:140753336   A>T            expect PASS      slippage test: no
  U2AF1_S34F         0.

In [48]:
%%bash
set -euo pipefail
source "$SHEET_SH"

for T in $(truth_sets); do
  D="$TRUTH_DIR/$T"
  [ -s "$D/variants.tsv" ] || { echo "No $D/variants.tsv: run the cell above first."; exit 1; }
  # As notebook 02 step 2: a minimal VCF (VAF and NAME in INFO), sorted, then
  # normalised against the reference. --check-ref e stops on a REF mismatch.
  {
    printf '##fileformat=VCFv4.2\n'
    printf '##INFO=<ID=VAF,Number=1,Type=Float,Description="Target allele fraction">\n'
    printf '##INFO=<ID=NAME,Number=1,Type=String,Description="Variant name">\n'
    awk '{printf "##contig=<ID=%s,length=%s>\n", $1, $2}' "$GENOME"
    printf '#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n'
    awk 'BEGIN{OFS="\t"} {print $1,$2,".",$3,$4,".","PASS","VAF="$5";NAME="$6}' "$D/variants.tsv"
  } > "$D/spike.unsorted.vcf"
  bcftools sort "$D/spike.unsorted.vcf" -Ov -o "$D/spike.sorted.vcf"
  echo "Truth set $T, bcftools norm:"
  bcftools norm -f "$REF" --check-ref e -m -both "$D/spike.sorted.vcf" -Ov -o "$D/spike.vcf"
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\t%INFO/NAME\n' "$D/spike.vcf" > "$D/variants.norm.tsv"
  echo "Truth set $T: $(grep -vc '^#' "$D/spike.vcf") records in spike.vcf (expect $(wc -l < "$D/variants.tsv"))"
done

Truth set B, bcftools norm:
Truth set B: 21 records in spike.vcf (expect 21)
Truth set C, bcftools norm:
Truth set C: 11 records in spike.vcf (expect 11)


Writing to /tmp/bcftools.cEfm15
Merging 1 temporary files
Done
Cleaning
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	21/0/0/0/0/0/0
Writing to /tmp/bcftools.MQONBK
Merging 1 temporary files
Done
Cleaning
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	11/0/0/0/0/0/0


Step 4-5 Simulate and Align

In [49]:
%%bash
set -euo pipefail
source "$SHEET_SH"
REDO=false   # true: simulate every negative again

for S in $(samples negative); do
  OUT1="$SIM_DIR/${S}_1.fq.gz"
  OUT2="$SIM_DIR/${S}_2.fq.gz"
  DONE="$SIM_DIR/${S}.done"
  if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$OUT1" ] && [ -s "$OUT2" ]; then
    echo "${S}: FASTQs already present, skipped."
    continue
  fi
  rm -f "$DONE"
  SECONDS=0
  SEED=$(get "$S" art_seed)
  echo "${S}: simulating, ART seed ${SEED} (about 5 minutes)..."
  # Notebook 02 step 4's command; only the seed and the names change.
  art_illumina -ss HSXt -p -l 150 -f 1000 -m 250 -s 50 -rs "$SEED" -na \
    -i "$PANEL_SRC" -o "$SCRATCH/${S}_" > "$RUN_LOGS/step4_art_${S}.log"
  bgzip -@ "$THREADS" -c "$SCRATCH/${S}_1.fq" > "$OUT1"
  bgzip -@ "$THREADS" -c "$SCRATCH/${S}_2.fq" > "$OUT2"
  rm -f "$SCRATCH/${S}_1.fq" "$SCRATCH/${S}_2.fq"
  touch "$DONE"
  echo "${S}: done in $((SECONDS / 60)) min, $(zcat "$OUT1" | awk 'NR % 4 == 1' | wc -l) read pairs (run 1's A: 3490000)."
done

A: simulating, ART seed 42 (about 5 minutes)...
A: done in 4 min, 3490000 read pairs (run 1's A: 3490000).
A2: simulating, ART seed 43 (about 5 minutes)...
A2: done in 3 min, 3490000 read pairs (run 1's A: 3490000).


In [50]:
%%bash
set -euo pipefail
source "$SHEET_SH"
REDO=false   # true: align every negative again

for S in $(samples negative); do
  BAM="$BAM_DIR/${S}.bam"
  DONE="$BAM_DIR/${S}.done"
  if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$BAM" ]; then
    echo "${S}: BAM already present, skipped."
    continue
  fi
  for f in "$SIM_DIR/${S}_1.fq.gz" "$SIM_DIR/${S}_2.fq.gz"; do
    [ -s "$f" ] || { echo "Missing $f: run step 4 first."; exit 1; }
  done
  rm -f "$DONE"
  SECONDS=0
  # Notebook 02 step 5's commands, with this sample's read group. Step 8 reuses
  # this BAM for the negatives, so the bwa command must match step 8's exactly.
  echo "${S}: aligning (about 17 minutes)..."
  bwa mem -t "$THREADS" -K 100000000 -R "@RG\tID:${S}\tSM:${S}\tPL:ILLUMINA" \
    "$REF" "$SIM_DIR/${S}_1.fq.gz" "$SIM_DIR/${S}_2.fq.gz" 2> "$RUN_LOGS/step5_bwa_${S}.log" |
    samtools sort -@ "$THREADS" -T "$SCRATCH/sort${S}" -o "$BAM"
  samtools index "$BAM"
  touch "$DONE"
  mosdepth -t "$THREADS" -n -b "$TARGETS_BED" "$QC_DIR/${S}" "$BAM"
  picard "$JAVA_MEM" CollectHsMetrics -I "$BAM" -O "$QC_DIR/${S}.hs_metrics.txt" -R "$REF" \
    -BI "$BAITS_IL" -TI "$TARGETS_IL" 2> "$RUN_LOGS/step5_hsmetrics_${S}.log"
  MEAN=$(awk '$1 == "total_region" {print $4}' "$QC_DIR/${S}.mosdepth.summary.txt")
  echo "${S}: done in $((SECONDS / 60)) min, mean target depth ${MEAN} (run 1's A: 1001.41)."
done

A: aligning (about 17 minutes)...
A: done in 22 min, mean target depth 1001.41 (run 1's A: 1001.41).
A2: aligning (about 17 minutes)...
A2: done in 22 min, mean target depth 1001.40 (run 1's A: 1001.41).


[bam_sort_core] merging from 1 files and 2 in-memory blocks...
[bam_sort_core] merging from 1 files and 2 in-memory blocks...


Step 6: spike each positive and check every variant

In [51]:
%%bash
set -euo pipefail
source "$SHEET_SH"
REDO=false   # true: spike every positive again
FAIL=0

for S in $(samples positive); do
  BASE=$(get "$S" base)
  T=$(get "$S" truth)
  SEED=$(get "$S" spike_seed)
  D="$TRUTH_DIR/$T"
  OUT="$BAM_DIR/${S}.spiked.bam"
  DONE="$BAM_DIR/${S}.done"
  TRUTH_VCF="$D/${S}.truth.vcf.gz"
  if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$OUT" ] && [ -s "$TRUTH_VCF" ]; then
    echo "${S}: spiked BAM already present, skipped."
    continue
  fi
  [ -s "$BAM_DIR/${BASE}.bam" ] || { echo "No ${BASE}.bam: run step 5 first (BAMs don't survive a session)."; exit 1; }
  [ -s "$D/spike.vcf" ] || { echo "No $D/spike.vcf: run step 2 first."; exit 1; }
  rm -f "$DONE"
  SECONDS=0
  N=$(grep -vc '^#' "$D/spike.vcf")

  # 1) Spike truth set T into the base sample's reads: notebook 02 step 6's
  #    command (its comments explain each option), with this sample's seed.
  #    The log goes to a file; its last lines are shown if BAMSurgeon fails.
  cd "$SCRATCH"
  rm -rf "${SCRATCH:?}"/bamsurgeon.tmp* "${SCRATCH:?}"/bamsurgeon_logs_*   # leftovers from an earlier try
  LOG="$RUN_LOGS/step6_bamsurgeon_${S}.log"
  echo "${S}: spiking ${N} variants (truth set ${T}) into ${BASE}, seed ${SEED} (roughly 20-40 minutes)..."
  if ! bamsurgeon -v "$D/spike.vcf" -f "$BAM_DIR/${BASE}.bam" -r "$REF" -o "$SCRATCH/${S}.unsorted.bam" \
      --aligner mem --alignerthreads "$THREADS" -p 1 --seed "$SEED" \
      --tmpdir "$SCRATCH/bamsurgeon.tmp" --vcf "$D/" > "$LOG" 2>&1; then
    tail -n 20 "$LOG"
    echo "${S}: BAMSurgeon failed (log above)."
    exit 1
  fi
  samtools sort -@ "$THREADS" -T "$SCRATCH/sort${S}" -o "$OUT" "$SCRATCH/${S}.unsorted.bam"
  samtools index "$OUT"
  rm -f "$SCRATCH/${S}.unsorted.bam"
  TRUTH_RAW="$D/${S}.unsorted.bamsurgeon.spike.vcf"

  # 2) BAMSurgeon's own check, kept as a record only: it misjudges large
  #    indels (notebook 02 step 6), and step 3 below replaces it.
  python3 "$BAMSURGEON_DIR/scripts/validate_spikein.py" --vcf "$TRUTH_RAW" \
    --bam "$OUT" --orig-bam "$BAM_DIR/${BASE}.bam" --ref "$REF" \
    --tsv "$QC_DIR/${S}.spikein_report.tsv" > "$RUN_LOGS/step6_validate_${S}.log" 2>&1 || true

  # 3) The decision, for every variant (new in notebook 03). For each row of
  #    step 2's junctions.tsv, count reads whose raw sequence (soft-clipped
  #    bases included) holds the k-mer only the mutant allele has, and reads
  #    holding the one only the reference allele has. Either strand counts.
  #    -F 0x900 skips secondary and supplementary records.
  #    Pass: at least 3 carrying reads, and the carrying count within 40% of
  #    the expected count or within 3 standard deviations of it, whichever is
  #    wider. The second allowance matters only at low VAF, where a handful of
  #    reads carry the variant and counting noise alone exceeds 40%.
  JFILE="$QC_DIR/${S}.spikein_junctions.tsv"
  printf 'name\tcarrying\twild_type\tvaf\trequested\tstatus\n' > "$JFILE"
  while IFS=$'\t' read -r NAME REGION MUT WT MUT_RC WT_RC VAF; do
    samtools view -F 0x900 "$OUT" "$REGION" < /dev/null |
      awk -v n="$NAME" -v r="$VAF" -v m1="$MUT" -v m2="$MUT_RC" -v w1="$WT" -v w2="$WT_RC" '
        index($10, m1) || index($10, m2) { m++; next }
        index($10, w1) || index($10, w2) { w++ }
        END {
          t = m + w; v = (t > 0) ? m / t : 0
          e = t * r; sd = sqrt(t * r * (1 - r))
          allow = (0.4 * e > 3 * sd) ? 0.4 * e : 3 * sd
          ok = (m >= 3 && m - e <= allow && e - m <= allow)
          printf "%s\t%d\t%d\t%.4f\t%s\t%s\n", n, m, w, v, r, (ok ? "OK" : "OUT_OF_RANGE") }' >> "$JFILE"
  done < <(tail -n +2 "$D/junctions.tsv")
  echo "${S}: read counts per variant:"
  cat "$JFILE"

  # 4) Sort, compress and index the truth VCF, as notebook 02 step 6.
  { grep '^#' "$TRUTH_RAW"; grep -v '^#' "$TRUTH_RAW" | sort -k1,1 -k2,2n; } | bgzip -c > "$TRUTH_VCF"
  tabix -f -p vcf "$TRUTH_VCF"
  RECORDS=$(bcftools view -H "$TRUTH_VCF" | wc -l)

  BAD=$(awk -F'\t' 'NR > 1 && $6 != "OK"' "$JFILE")
  if [ -n "$BAD" ] || [ "$RECORDS" -ne "$N" ]; then
    echo "${S}: check failed (truth records ${RECORDS} of ${N}; rows not OK above). ${S}.done not written."
    FAIL=1
    continue   # go on to the next positive; the cell fails at the end
  fi
  touch "$DONE"
  echo "${S}: done in $((SECONDS / 60)) min. Truth records: ${RECORDS} of ${N}. All read counts OK."
done
[ "$FAIL" -eq 0 ] || exit 1

B: spiking 21 variants (truth set B) into A, seed 42 (roughly 20-40 minutes)...
B: read counts per variant:
name	carrying	wild_type	vaf	requested	status
NRAS_G12D	22	932	0.0231	0.02	OK
KRAS_G12D	19	937	0.0199	0.02	OK
FLT3_ITD30	44	826	0.0506	0.05	OK
FLT3_T227M	425	422	0.5018	0.5	OK
IDH2_R140Q	37	743	0.0474	0.05	OK
TP53_R273H	340	499	0.4052	0.4	OK
TP53_P72R	435	438	0.4983	0.5	OK
SRSF2_P95H	95	847	0.1008	0.1	OK
CALR_type1	61	599	0.0924	0.1	OK
DNMT3A_R882H	20	951	0.0206	0.02	OK
SF3B1_K700E	281	479	0.3697	0.4	OK
IDH1_R132H	102	841	0.1082	0.1	OK
ASXL1_G646Wfs12	290	446	0.3940	0.4	OK
U2AF1_S34F	49	959	0.0486	0.05	OK
MYD88_L265P	81	698	0.1040	0.1	OK
RHOA_G17V	18	918	0.0192	0.02	OK
TET2_I1762V	392	396	0.4975	0.5	OK
NPM1_typeA	17	853	0.0195	0.02	OK
DDX41_D140Gfs2	375	391	0.4896	0.5	OK
BRAF_V600E	52	919	0.0536	0.05	OK
JAK2_V617F	403	634	0.3886	0.4	OK
B: done in 16 min. Truth records: 21 of 21. All read counts OK.
C: spiking 11 variants (truth set C) into A2, seed 43 (roughly 20-40 minutes)...
C:

[bam_sort_core] merging from 1 files and 2 in-memory blocks...
[bam_sort_core] merging from 1 files and 2 in-memory blocks...


Step 7: FASTQs for each positive

In [52]:
%%bash
set -euo pipefail
source "$SHEET_SH"
REDO=false   # true: write every positive's FASTQs again

for S in $(samples positive); do
  BASE=$(get "$S" base)
  OUT1="$SIM_DIR/${S}_1.fq.gz"
  OUT2="$SIM_DIR/${S}_2.fq.gz"
  DONE="$SIM_DIR/${S}.done"
  if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$OUT1" ] && [ -s "$OUT2" ]; then
    echo "${S}: FASTQs already present, skipped."
    continue
  fi
  # Only a spiked BAM that passed step 6's check (its .done marker).
  [ -s "$BAM_DIR/${S}.spiked.bam" ] && [ -f "$BAM_DIR/${S}.done" ] ||
    { echo "No checked ${S}.spiked.bam: run step 6 first (in this session)."; exit 1; }
  rm -f "$DONE"
  # Notebook 02 step 7's commands.
  samtools collate -@ "$THREADS" -u -O -T "$SCRATCH/collate${S}" "$BAM_DIR/${S}.spiked.bam" |
    samtools fastq -@ "$THREADS" -n -1 "$OUT1" -2 "$OUT2" -0 /dev/null -s /dev/null 2> "$RUN_LOGS/step7_fastq_${S}.log"
  touch "$DONE"
  echo "${S}: read pairs ${BASE} $(zcat "$SIM_DIR/${BASE}_1.fq.gz" | awk 'NR % 4 == 1' | wc -l), ${S} $(zcat "$OUT1" | awk 'NR % 4 == 1' | wc -l) (should match)"
done

B: read pairs A 3490000, B 3490000 (should match)
C: read pairs A2 3490000, C 3490000 (should match)


Step 8: the pipeline on every sample

In [54]:
%%bash
set -euo pipefail
source "$SHEET_SH"
REDO=false               # true: rerun the pipeline on every sample
REUSE_STEP5=true         # negatives: reuse step 5's alignment while it's here

for S in $(samples negative) $(samples positive); do
  DONE="$CALLS_DIR/${S}.done"
  if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$CALLS_DIR/${S}.pass.vcf.gz" ]; then
    echo "${S}: calls already present, skipped."
    continue
  fi
  for f in "$SIM_DIR/${S}_1.fq.gz" "$SIM_DIR/${S}_2.fq.gz"; do
    [ -s "$f" ] || { echo "Missing $f: run step 4 (negatives) or step 7 (positives) first."; exit 1; }
  done
  rm -f "$DONE"
  SECONDS=0

  # The test pipeline: notebook 02 step 8's commands, unchanged. Mutect2 and
  # FilterMutectCalls keep their defaults on purpose (see "About").
  # 1) Align and sort (or reuse step 5's identical alignment for a negative).
  if [ "$(get "$S" kind)" = negative ] && [ "$REUSE_STEP5" = true ] && [ -f "$BAM_DIR/${S}.done" ]; then
    echo "${S}: reusing step 5's alignment."
    ALN="$BAM_DIR/${S}.bam"
  else
    echo "${S}: aligning (about 25 minutes)..."
    ALN="$BAM_DIR/${S}.aln.bam"
    bwa mem -t "$THREADS" -K 100000000 -R "@RG\tID:${S}\tSM:${S}\tPL:ILLUMINA" \
      "$REF" "$SIM_DIR/${S}_1.fq.gz" "$SIM_DIR/${S}_2.fq.gz" 2> "$RUN_LOGS/step8_bwa_${S}.log" |
      samtools sort -@ "$THREADS" -T "$SCRATCH/sort${S}" -o "$ALN"
  fi
  # 2) Mark duplicates.
  gatk --java-options "$JAVA_MEM" MarkDuplicates -I "$ALN" -O "$BAM_DIR/${S}.dedup.bam" \
    -M "$QC_DIR/${S}.dup_metrics.txt" --CREATE_INDEX true --TMP_DIR "$SCRATCH" 2> "$RUN_LOGS/step8_markdup_${S}.log"
  # 3) Call, tumour-only, inside the targets padded by 50 bp.
  gatk --java-options "$JAVA_MEM" Mutect2 -R "$REF" -I "$BAM_DIR/${S}.dedup.bam" \
    -L "$TARGETS_BED" -ip 50 --native-pair-hmm-threads "$THREADS" --tmp-dir "$SCRATCH" \
    -O "$CALLS_DIR/${S}.unfiltered.vcf.gz" 2> "$RUN_LOGS/step8_mutect2_${S}.log"
  # 4) Filter.
  gatk --java-options "$JAVA_MEM" FilterMutectCalls -R "$REF" -V "$CALLS_DIR/${S}.unfiltered.vcf.gz" \
    --tmp-dir "$SCRATCH" -O "$CALLS_DIR/${S}.filtered.vcf.gz" 2> "$RUN_LOGS/step8_filter_${S}.log"
  # 5) PASS calls, normalised like the truth set (as run 1).
  bcftools view -f PASS "$CALLS_DIR/${S}.filtered.vcf.gz" |
    bcftools norm -f "$REF" -m -both -Oz -o "$CALLS_DIR/${S}.pass.vcf.gz"
  tabix -f -p vcf "$CALLS_DIR/${S}.pass.vcf.gz"
  # New in notebook 03: every candidate, filtered or not, normalised the same
  # way, so step 9 can report each truth variant's filter and allele fraction.
  bcftools norm -f "$REF" -m -both "$CALLS_DIR/${S}.filtered.vcf.gz" -Oz \
    -o "$CALLS_DIR/${S}.filtered.norm.vcf.gz" 2> /dev/null
  tabix -f -p vcf "$CALLS_DIR/${S}.filtered.norm.vcf.gz"

  touch "$DONE"
  echo "${S}: done in $((SECONDS / 60)) min. Candidates by filter outcome:"
  bcftools query -f '%FILTER\n' "$CALLS_DIR/${S}.filtered.vcf.gz" | sort | uniq -c
done

A: reusing step 5's alignment.
Tool returned:
0
Tool returned:
SUCCESS
A: done in 8 min. Candidates by filter outcome:
A2: reusing step 5's alignment.
Tool returned:
0
Tool returned:
SUCCESS
A2: done in 8 min. Candidates by filter outcome:
B: aligning (about 25 minutes)...
Tool returned:
0
Tool returned:
SUCCESS
B: done in 33 min. Candidates by filter outcome:
      4 germline
     16 PASS
      1 slippage
C: aligning (about 25 minutes)...
Tool returned:
0
Tool returned:
SUCCESS
C: done in 32 min. Candidates by filter outcome:
      5 PASS
      1 slippage


Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	0/0/0/0/0/0/0
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	0/0/0/0/0/0/0
[bam_sort_core] merging from 1 files and 2 in-memory blocks...
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	16/0/0/0/0/0/0
[bam_sort_core] merging from 1 files and 2 in-memory blocks...
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	5/0/0/0/0/0/0


Step 9: score every sample

In [55]:
%%bash
set -euo pipefail
source "$SHEET_SH"
[ -d "$SDF" ] || { echo "No RTG SDF: run notebook 02 step 0 first."; exit 1; }

# 1) Negatives: every PASS call inside the targets is a false positive.
for S in $(samples negative); do
  [ -s "$CALLS_DIR/${S}.pass.vcf.gz" ] || { echo "Missing ${S}.pass.vcf.gz: run step 8 first."; exit 1; }
  echo "False positives, ${S} (negative): $(bcftools view -H -R "$TARGETS_BED" "$CALLS_DIR/${S}.pass.vcf.gz" | wc -l)"
done

# 2) Positives: vcfeval against that sample's truth VCF (notebook 02 step 9
#    explains the options), then false positives split by whether the base
#    sample has the same call.
for S in $(samples positive); do
  T=$(get "$S" truth)
  BASE=$(get "$S" base)
  TRUTH_VCF="$TRUTH_DIR/$T/${S}.truth.vcf.gz"
  for f in "$TRUTH_VCF" "$CALLS_DIR/${S}.pass.vcf.gz" "$CALLS_DIR/${S}.filtered.norm.vcf.gz"; do
    [ -s "$f" ] || { echo "Missing $f: run steps 6 and 8 first."; exit 1; }
  done
  # vcfeval refuses an existing output folder. ${VAR:?} stops the cell if the
  # variable is empty, so this can never remove anything above EVAL_DIR.
  rm -rf "${EVAL_DIR:?}/${S:?}"
  rtg vcfeval -b "$TRUTH_VCF" -c "$CALLS_DIR/${S}.pass.vcf.gz" -t "$SDF" \
    -e "$TARGETS_BED" --squash-ploidy -o "$EVAL_DIR/$S" > "$RUN_LOGS/step9_vcfeval_${S}.log"
  echo
  echo "== ${S} (truth set ${T}, built from ${BASE})"
  cat "$EVAL_DIR/$S/summary.txt"
  FP=$(bcftools view -H "$EVAL_DIR/$S/fp.vcf.gz" | wc -l)
  SHARED=$(bcftools isec -n=2 -w1 -c none "$EVAL_DIR/$S/fp.vcf.gz" "$CALLS_DIR/${BASE}.pass.vcf.gz" | awk '!/^#/' | wc -l)
  echo "False positives: ${FP} (${SHARED} also called in ${BASE}, $((FP - SHARED)) only in ${S})"

  # Two tables for the R script: vcfeval's verdict on each truth variant (with
  # the VAF BAMSurgeon reports), and every Mutect2 candidate with its filter
  # outcome and allele fraction (AF is per sample, hence the [ ]).
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\tfound\n'  "$EVAL_DIR/$S/tp-baseline.vcf.gz" >  "$EVAL_DIR/${S}_vcfeval.tsv"
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\tmissed\n' "$EVAL_DIR/$S/fn.vcf.gz"          >> "$EVAL_DIR/${S}_vcfeval.tsv"
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%FILTER\t[%AF]\n' "$CALLS_DIR/${S}.filtered.norm.vcf.gz" > "$EVAL_DIR/${S}_mutect2.tsv"
done

# 3) One row per truth variant, scored against the expectation step 2 wrote
#    down before the run. Base R; the script is saved beside the results.
cat > "$EVAL_DIR/results_table.R" <<'REOF'
options(width = 200)
eval_dir  <- Sys.getenv("EVAL_DIR")
truth_dir <- Sys.getenv("TRUTH_DIR")
qc_dir    <- Sys.getenv("QC_DIR")
sheet <- read.delim(Sys.getenv("SAMPLE_SHEET"), colClasses = "character")
positives <- sheet[sheet$kind == "positive", ]

# Headerless tab-separated file -> table with these column names (an empty
# file gives an empty table instead of an error).
read_plain <- function(path, cols) {
  if (file.size(path) == 0) {
    return(setNames(as.data.frame(matrix(character(0), ncol = length(cols))), cols))
  }
  read.delim(path, header = FALSE, col.names = cols, colClasses = "character")
}

results <- list()
for (i in seq_len(nrow(positives))) {
  s  <- positives$sample[i]
  ts <- positives$truth[i]
  truth <- read.delim(file.path(truth_dir, ts, "truth_set.tsv"), colClasses = "character")
  norm  <- read_plain(file.path(truth_dir, ts, "variants.norm.tsv"),
                      c("CHROM", "POS", "REF", "ALT", "VAF_target", "name"))
  ve    <- read_plain(file.path(eval_dir, paste0(s, "_vcfeval.tsv")),
                      c("CHROM", "POS", "REF", "ALT", "VAF_bamsurgeon", "status"))
  m2    <- read_plain(file.path(eval_dir, paste0(s, "_mutect2.tsv")),
                      c("CHROM", "POS", "REF", "ALT", "filter", "AF_mutect2"))
  jc    <- read.delim(file.path(qc_dir, paste0(s, ".spikein_junctions.tsv")), colClasses = "character")

  # Join on the normalised positions; Mutect2's candidates also on REF and ALT.
  m <- merge(norm, truth[, c("name", "expect", "slippage_test")], by = "name")
  m <- merge(m, ve[, c("CHROM", "POS", "VAF_bamsurgeon", "status")], by = c("CHROM", "POS"), all.x = TRUE)
  m <- merge(m, m2, by = c("CHROM", "POS", "REF", "ALT"), all.x = TRUE)
  m <- merge(m, data.frame(name = jc$name, VAF_reads = jc$vaf), by = "name", all.x = TRUE)

  m$found   <- m$status %in% "found"
  m$outcome <- ifelse(m$found, "PASS", ifelse(is.na(m$filter), "not called", m$filter))
  m$as_expected <- ifelse(m$expect == "limit", NA,
                          (m$expect == "PASS" & m$found) |
                          (m$expect == "germline" & grepl("germline", m$outcome)) |
                          (m$expect == "slippage" & grepl("slippage", m$outcome)))
  m$sample <- s

  cat("\n==", s, "(truth set", ts, ")\n")
  cols <- c("name", "VAF_target", "VAF_reads", "VAF_bamsurgeon", "AF_mutect2", "expect", "outcome", "as_expected")
  print(m[order(-as.numeric(m$VAF_target), m$name), cols], row.names = FALSE)
  scored <- m[m$expect != "limit", ]
  cat("\nAs expected:", sum(scored$as_expected), "of", nrow(scored), "\n")
  limit <- m[m$expect == "limit", ]
  if (nrow(limit) > 0) cat("Detection-limit tier (no expectation): found", sum(limit$found), "of", nrow(limit), "\n")
  cat("\nFound by tier:\n")
  print(aggregate(found ~ VAF_target, data = m, FUN = function(x) paste(sum(x), "of", length(x))), row.names = FALSE)
  results[[s]] <- m
}
all_rows <- do.call(rbind, results)
write.table(all_rows, file.path(eval_dir, "results_by_variant.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
cat("\nWrote", file.path(eval_dir, "results_by_variant.tsv"), "\n")
REOF
Rscript "$EVAL_DIR/results_table.R"

False positives, A (negative): 0
False positives, A2 (negative): 0

== B (truth set B, built from A)
Threshold  True-pos-baseline  True-pos-call  False-pos  False-neg  Precision  Sensitivity  F-measure
----------------------------------------------------------------------------------------------------
     None                 16             16          0          5     1.0000       0.7619     0.8649
False positives: 0 (0 also called in A, 0 only in B)

== C (truth set C, built from A2)
Threshold  True-pos-baseline  True-pos-call  False-pos  False-neg  Precision  Sensitivity  F-measure
----------------------------------------------------------------------------------------------------
     None                  5              5          0          6     1.0000       0.4545     0.6250
False positives: 0 (0 also called in A2, 0 only in C)

== B (truth set B )
            name VAF_target VAF_reads VAF_bamsurgeon AF_mutect2   expect  outcome as_expected
  DDX41_D140Gfs2        0.5    0.489

Reference sequence chr4 is used in baseline but not in calls.
Reference sequence chr13 is used in baseline but not in calls.
There were 16 variants not thresholded in ROC data files due to missing or invalid GQ (FORMAT) values.
Could not select maximized F-measure threshold from ROC data, only un-thresholded statistics will be shown. Consider selecting a different scoring attribute with --vcf-score-field
Reference sequence chr2 is used in baseline but not in calls.
Reference sequence chr4 is used in baseline but not in calls.
Reference sequence chr12 is used in baseline but not in calls.
Reference sequence chr17 is used in baseline but not in calls.
There were 5 variants not thresholded in ROC data files due to missing or invalid GQ (FORMAT) values.
Could not select maximized F-measure threshold from ROC data, only un-thresholded statistics will be shown. Consider selecting a different scoring attribute with --vcf-score-field
